# 13주차 실습 : 딥러닝 기반 형태소 분석·품사 태깅과 개체명 인식
> **2026711051 박민지**

### [1] 실습 진행 관련 설명

13주차 강의에서 배운 **딥러닝 기반 언어 분석**을 직접 구현해 본다.
실습은 크게 세 부분으로 구성되어 있다.

- **1부. 라이브러리를 이용한 방법** — KoNLPy 형태소 분석기로 품사 태깅 결과를 비교한다.
- **2부. 직접 함수를 구현하는 방법** — PyTorch로 Bi-LSTM 품사 태거를 직접 만든다.
- **3부. 응용** — 같은 구조를 개체명 인식(NER)에 적용하고 **개체 단위 F1**으로 평가한다.

처음에는 라이브러리를 이용한 방법을 실습하고, 그 이후에 직접 모델을 구현하여 실습을 진행한다.

### [2] 코드 구현의 흐름

- (1) (데이터 측면) 데이터 불러오기 / 만들기
- (2) (전처리) 어휘 사전 구축 → 정수 인코딩 → 패딩
- (3) (모델 측면) Bi-LSTM 태거 정의하기
- (4) (학습 과정) 손실 함수와 최적화 기법으로 학습하기
- (5) (성능 평가) 토큰 단위 정확도와 개체 단위 F1으로 평가하기

### [3] 과제 안내

- 코드 셀 안의 `raise NotImplementedError(...)` 부분을 지우고 직접 구현하여 노트북을 완성한다.
- `[서술형]` 문항은 바로 아래 마크다운 셀에 직접 답을 작성한다.
- 모든 셀을 **위에서부터 순서대로** 실행한 뒤, 실행 결과가 포함된 `.ipynb` 파일을 제출한다.
- 평가 : 코드 완성도 40% / 결과 해석 30% / 서술형 답안 30%

In [1]:
# [0] 환경 설정  ─ 런타임을 새로 시작했다면 이 셀을 가장 먼저 실행한다.
#     (Windows VS Code 실행) Colab 전용 apt-get(JDK·나눔 폰트 설치)은 뺐다.
#     · KoNLPy 는 자바(JVM)가 있어야 동작하므로 JDK 를 미리 설치해 두었다.
#         winget install --id Microsoft.OpenJDK.17 -e
#     · JAVA_HOME 이 비어 있으면 설치된 JDK 폴더를 찾아 지정한다.
#     · 한글 폰트는 Windows 기본 글꼴 Malgun Gothic 을 쓴다.
import os
import glob

if not os.environ.get('JAVA_HOME'):
    jdks = sorted(glob.glob(r'C:\Program Files\Microsoft\jdk-*') +
                  glob.glob(r'C:\Program Files\Java\jdk*') +
                  glob.glob(r'C:\Program Files\Eclipse Adoptium\jdk*'))
    if jdks:
        os.environ['JAVA_HOME'] = jdks[-1]
print('JAVA_HOME :', os.environ.get('JAVA_HOME', '(없음 — 1부는 건너뜀)'))

import matplotlib
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
print('한글 폰트 설정 완료 :', plt.rcParams['font.family'])


JAVA_HOME : C:\Program Files\Microsoft\jdk-17.0.21.102-hotspot


한글 폰트 설정 완료 : ['Malgun Gothic']


## (공통) 환경 설정

Google Colab 기준이며, 별도의 GPU 없이도 몇 분 안에 학습이 끝난다.
`konlpy`는 자바(JDK)에 의존하므로 설치에 1~2분 정도 걸린다.
설치에 실패하더라도 2부·3부 실습은 그대로 진행할 수 있다.

In [2]:
# 1부(KoNLPy) 실습을 위한 설치 — 최초 1회만 실행 (노트북 커널의 파이썬에 설치된다)
#   JPype1 은 konlpy 와 함께 Python 3.13 에 맞는 최신판이 설치된다.
#   (Colab 원본의 JPype1==1.4.1 고정은 Python 3.13 에서 쓰지 않는다)
%pip install -q konlpy


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import random
import numpy as np
import torch
import torch.nn as nn

# (재현성) 무작위 시드 고정 — 과제 제출 시 반드시 유지할 것
SEED = 13
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch version :', torch.__version__)
print('device        :', DEVICE)

torch version : 2.14.1+cpu
device        : cpu


---
# 1부. KoNLPy를 이용한 형태소 분석과 품사 태깅

**목적** : 오픈 라이브러리인 KoNLPy의 여러 형태소 분석기를 이용하여 품사 태깅 결과를 비교한다.

강의에서 살펴본 것처럼 한국어는 교착어이기 때문에 어절을 그대로 사용하면
어간·접사·어미가 뭉쳐 있어 의미를 파악하기 어렵다.
따라서 형태소 단위로 나누고 각 형태소에 품사(POS) 태그를 붙이는 작업이 필요하다.

> **[설명]** 형태소 분석은 어절을 뜻을 가진 가장 작은 단위(형태소)로 쪼개고, 품사 태깅은 쪼갠 조각마다 품사 이름을 붙이는 작업이다. 한국어는 '갔다 = 가 + 았 + 다'처럼 한 어절에 여러 형태소가 붙어 있어서, 검색·분류·개체명 인식 같은 거의 모든 작업의 첫 단계로 쓴다. 분석기마다 사전과 태그 체계가 달라 결과도 달라지므로, 같은 문장을 세 분석기에 넣고 비교한다.

**Okt · Komoran · Hannanum**: KoNLPy(`konlpy.tag`)의 형태소 분석기 클래스로, 내부에서 자바(JVM)로 만든 분석기를 불러 쓴다.
- `Okt()`: 트위터 한국어 처리기를 이은 분석기. 구어체·신조어에 강하고 태그가 `Noun`, `Josa`처럼 단순하다.
- `Komoran()`: 세종 태그셋(`NNG`, `JKS`, `EP` 등)을 쓰는 분석기. 사전 기반이라 문어체 분석이 정교하다.
- `Hannanum()`: KAIST 태그셋(`N`, `J`, `P`, `E` 등)을 쓰는 분석기. 기본 `pos()`는 9개의 굵은 태그를 붙인다.
- `tagger.pos(문장)`: `[(형태소, 품사), ...]` 리스트를 돌려준다. / `morphs()`는 형태소만, `nouns()`는 명사만 돌려준다.

**pos 결과 읽는 법**
- 같은 문장이라도 분석기마다 나누는 단위와 태그 이름이 다르다. 예: 주격 조사는 Okt `Josa`, Komoran `JKS`, Hannanum `J`.

In [4]:
# (1-1) KoNLPy 형태소 분석기 준비
#   JDK 설치가 실패하면 KoNLPy 는 동작하지 않는다. 그 경우 1부는 건너뛰고 2부부터 진행한다.

KONLPY_OK = False
taggers = {}
try:
    from konlpy.tag import Okt, Komoran, Hannanum
    taggers = {'Okt': Okt(), 'Komoran': Komoran(), 'Hannanum': Hannanum()}
    _ = taggers['Okt'].pos('설치 확인')          # 실제로 JVM 이 뜨는지 확인
    KONLPY_OK = True
    print('KoNLPy 준비 완료 :', list(taggers))
except Exception as e:
    print('[안내] KoNLPy 를 사용할 수 없습니다 — 1부를 건너뛰고 2부부터 진행하세요.')
    print('       원인 :', type(e).__name__, e)


KoNLPy 준비 완료 : ['Okt', 'Komoran', 'Hannanum']


### [TODO 1-1] 분석기 비교

아래 세 문장을 세 분석기로 각각 품사 태깅하고 결과를 출력하시오.
`raise NotImplementedError(...)` 를 지우고 직접 구현한다.

In [5]:
# (1-2) 세 분석기의 품사 태깅 결과 비교

sentences = [
    '나는 어제 친구랑 부산에 갔다',
    '아버지가방에들어가신다',          # 띄어쓰기가 없는 경우
    '이 영화 진짜 꿀잼이었음ㅋㅋ',      # 구어체·신조어
]


def tag_sentence(tagger, s):
    """tagger 로 문장 s 를 품사 태깅한 결과를 돌려주시오."""
    # 실습 1-2 : tagger.pos(s) 로 태깅 결과를 반환하시오.
    return tagger.pos(s)


if not KONLPY_OK:
    print('[건너뜀] KoNLPy 를 사용할 수 없어 1부를 건너뜁니다.')
else:
    for s in sentences:
        print('=' * 70)
        print('문장 :', s)
        for name, tagger in taggers.items():
            try:
                print(f'  {name:8s}', tag_sentence(tagger, s))
            except NotImplementedError as e:
                print('  ▶ 미구현 :', e)
                break


문장 : 나는 어제 친구랑 부산에 갔다


  Okt      [('나', 'Noun'), ('는', 'Josa'), ('어제', 'Noun'), ('친구', 'Noun'), ('랑', 'Josa'), ('부산', 'Noun'), ('에', 'Josa'), ('갔다', 'Verb')]
  Komoran  [('나', 'NP'), ('는', 'JX'), ('어제', 'MAG'), ('친구', 'NNG'), ('랑', 'JKB'), ('부산', 'NNP'), ('에', 'JKB'), ('가', 'VV'), ('았', 'EP'), ('다', 'EC')]


  Hannanum [('나', 'N'), ('는', 'J'), ('어제', 'N'), ('친구', 'N'), ('랑', 'J'), ('부산', 'N'), ('에', 'J'), ('가', 'P'), ('아다', 'E')]
문장 : 아버지가방에들어가신다
  Okt      [('아버지', 'Noun'), ('가방', 'Noun'), ('에', 'Josa'), ('들어가신다', 'Verb')]
  Komoran  [('아버지', 'NNG'), ('가방', 'NNP'), ('에', 'JKB'), ('들어가', 'VV'), ('시', 'EP'), ('ㄴ다', 'EC')]
  Hannanum [('아버지가방에들어가', 'N'), ('이', 'J'), ('시ㄴ다', 'E')]
문장 : 이 영화 진짜 꿀잼이었음ㅋㅋ
  Okt      [('이', 'Noun'), ('영화', 'Noun'), ('진짜', 'Noun'), ('꿀', 'Noun'), ('잼이었음', 'Verb'), ('ㅋㅋ', 'KoreanParticle')]
  Komoran  [('이', 'MM'), ('영화', 'NNG'), ('진짜', 'MAG'), ('꿀잼이었음ㅋㅋ', 'NA')]
  Hannanum [('이', 'M'), ('영화', 'N'), ('진짜', 'N'), ('꿀잼이었음ㅋㅋ', 'N')]


> **[결과]** KoNLPy의 `Okt`, `Komoran`, `Hannanum` 세 분석기로 문장 3개를 태깅하고, `tag_sentence` 함수에서 `tagger.pos(s)`를 불렀다.
> - 평범한 문장에서는 세 분석기 모두 '나/는/어제/친구/랑/부산/에'까지 같은 자리에서 끊었다. 다만 `갔다`를 Okt는 `Verb` 한 덩어리로 두고, Komoran은 `가/VV + 았/EP + 다/EC`로, Hannanum은 `가/P + 아다/E`로 나누었다.
> - 띄어쓰기가 없는 `아버지가방에들어가신다`에서는 셋 다 '아버지가 방에'로 읽지 못했다. Okt와 Komoran은 `가방`을 명사로 잘랐고, Hannanum은 `아버지가방에들어가`를 통째로 명사 하나로 보았다.
> - 구어체 `꿀잼이었음ㅋㅋ`는 Komoran이 `NA`(분석 불능), Hannanum이 명사 한 덩어리로 처리했고, Okt만 `ㅋㅋ`를 `KoreanParticle`로 떼어 냈다.

### [서술형 1-2]

위 결과를 보고 다음 물음에 답하시오. (각 2~3문장)

1. 띄어쓰기가 없는 문장에서 분석기별 결과가 어떻게 달라졌는가?
2. 구어체·신조어 문장에서 가장 좋은 결과를 보인 분석기는 무엇이며, 그 이유는 무엇이라고 생각하는가?
3. 세 분석기의 품사 태그 체계(태그셋)가 서로 다른 것이 실제 서비스 개발에서 어떤 문제를 일으킬 수 있는가?


**서술형 1-2 답안**

**[1] 띄어쓰기가 없는 문장** 세 분석기 모두 '아버지가 방에 들어가신다'라는 뜻을 살리지 못했지만, 틀린 방식은 달랐다. Okt와 Komoran은 `아버지/가방/에`로 잘라 '아버지 가방에'라는 엉뚱한 뜻을 만들었지만, 동사 `들어가`는 찾아냈다. Hannanum은 `아버지가방에들어가`를 명사 하나로 묶어 버려서 사실상 분석에 실패했다. '가'를 조사로 떼어 내는 대신 사전에 있는 명사 '가방'을 먼저 잡은 것으로 보이며, 띄어쓰기가 없으면 어절 경계라는 가장 큰 단서가 사라져 분석기마다 결과가 크게 갈린다는 점을 확인했다.

**[2] 구어체·신조어 문장** Okt가 가장 나았다. Komoran은 `꿀잼이었음ㅋㅋ`를 `NA`로 포기했고 Hannanum은 명사로 뭉뚱그렸지만, Okt는 `꿀`, `잼이었음`, `ㅋㅋ`로 나누고 `ㅋㅋ`에 `KoreanParticle`이라는 별도 태그를 붙였다. Okt는 트위터 한국어 처리기에서 출발해 SNS 문장을 다루도록 만들어졌고, 반복 자음 같은 구어 표현을 따로 처리하는 규칙이 있기 때문이라고 생각한다. 다만 Okt도 `꿀잼`을 한 단어로 보지 못했으니, 신조어는 사용자 사전을 더해야 제대로 잡을 수 있다.

**[3] 태그 체계가 다를 때의 문제** 같은 주격 조사를 Okt는 `Josa`, Komoran은 `JKS`, Hannanum은 `J`로 부른다. 그래서 'NNG만 골라 키워드로 쓴다'처럼 태그 이름에 기대어 짠 코드는 분석기를 바꾸는 순간 아무것도 뽑지 못한다. 세분화 수준도 달라서, Okt는 주격·부사격·보조사를 모두 `Josa` 하나로 묶고 `갔다`도 쪼개지 않기 때문에 Komoran 기준으로 만든 규칙이나 학습 데이터를 그대로 옮겨 쓸 수 없다. 실제 서비스에서는 처음에 분석기 하나를 정해 사전 구축·학습·추론 전 과정에서 같은 것을 쓰고, 여러 분석기를 섞어야 한다면 공통 태그(예: Universal POS)로 바꾸는 대응표를 먼저 만들어야 한다.

---
# 2부. Bi-LSTM 품사 태거 직접 구현

**목적** : 강의에서 살펴본 `임베딩 → Bi-LSTM → 분류기` 구조를 PyTorch로 직접 구현한다.

학습 데이터로는 세종 말뭉치와 같은 실제 코퍼스를 쓰는 것이 이상적이지만,
용량과 저작권 문제가 있으므로 이번 실습에서는 **한국어 문장 생성기**로 만든
형태소·품사 데이터를 사용한다. 태그는 세종 태그셋을 따른다.

| 태그 | 의미 | 예 |
|---|---|---|
| NNG | 일반 명사 | 학생, 도서관 |
| NP  | 대명사 | 나, 너 |
| JKS | 주격 조사 | 이, 가 |
| JKO | 목적격 조사 | 을, 를 |
| JKB | 부사격 조사 | 에, 에서 |
| VV  | 동사 | 먹, 읽, 가 |
| EP  | 선어말 어미 | 었, 겠 |
| EF  | 종결 어미 | 다, 요 |
| JX  | 보조사 | 은, 는 |
| SF  | 마침표 | . |

**주의** : 이 데이터에는 일부러 **중의적인 형태소**를 넣어 두었다.
예를 들어 `'가'`는 주격 조사(JKS)일 수도 있고 동사 어간(VV, "가/VV + 았/EP + 다/EF → 갔다")일 수도 있다.
따라서 모델은 앞뒤 문맥을 보고 태그를 결정해야 한다.

> **[설명]** Bi-LSTM 품사 태거는 형태소마다 '앞에서부터 읽은 문맥'과 '뒤에서부터 읽은 문맥'을 함께 보고 품사를 고르는 모델이다. 규칙이나 사전 없이 데이터에서 문맥 단서를 스스로 배우므로, `'가'`처럼 조사도 되고 동사도 되는 중의적인 형태소를 구분하는 데 쓴다. 이 절에서는 문장 생성기로 만든 세종 태그셋 데이터로 `임베딩 → Bi-LSTM → 선형 분류기`를 직접 조립한다.

In [6]:
# (1) (데이터 측면) 품사 태깅 학습 데이터 만들기

SUBJ  = ['학생', '선생님', '친구', '동생', '기자', '의사', '아이', '선수']
PRON  = ['나', '너', '우리']
PLACE = ['학교', '도서관', '공원', '시장', '병원', '회사', '바다', '식당']
OBJ   = ['밥', '책', '커피', '사과', '영화', '노래', '편지', '사진']
VERB  = ['먹', '읽', '보', '사', '쓰']          # 목적어를 취하는 동사
JKB_L = ['에', '에서']
EP_L  = ['었', '겠']


def has_jong(word):
    """마지막 음절에 받침이 있는지 확인 (한글 유니코드 계산)"""
    code = ord(word[-1]) - 0xAC00
    return 0 <= code <= 11171 and code % 28 != 0


def josa(word, with_jong, without_jong):
    """받침 유무에 따라 조사를 선택한다. 예) 학생+이 / 친구+가"""
    return with_jong if has_jong(word) else without_jong


# 형태소는 모두 기본형으로 표기한다. (세종 말뭉치 방식)
#   갔다 → 가/VV + 았/EP + 다/EF  (본 실습 데이터는 단순화를 위해 '었'만 사용한다)

def make_pos_sentence():
    """(형태소, 품사) 쌍의 리스트를 하나 생성한다."""
    r = random.random()
    if r < 0.35:
        # 유형 A : 주어 + 장소 + 목적어 + 동사   (예: 학생이 도서관에서 책을 읽었다)
        s, p, o = random.choice(SUBJ), random.choice(PLACE), random.choice(OBJ)
        seq = [(s, 'NNG'), (josa(s, '이', '가'), 'JKS'),
               (p, 'NNG'), (random.choice(JKB_L), 'JKB'),
               (o, 'NNG'), (josa(o, '을', '를'), 'JKO'),
               (random.choice(VERB), 'VV'), (random.choice(EP_L), 'EP'),
               ('다', 'EF'), ('.', 'SF')]
    elif r < 0.6:
        # 유형 B : 대명사 주어 + 목적어 + 동사   (예: 나는 커피를 사겠다)
        n, o = random.choice(PRON), random.choice(OBJ)
        seq = [(n, 'NP'), (josa(n, '은', '는'), 'JX'),
               (o, 'NNG'), (josa(o, '을', '를'), 'JKO'),
               (random.choice(VERB), 'VV'), (random.choice(EP_L), 'EP'),
               ('다', 'EF'), ('.', 'SF')]
    elif r < 0.85:
        # 유형 C : '가'가 동사 어간으로 쓰이는 경우 (예: 친구가 학교에 갔다)
        s, p = random.choice(SUBJ), random.choice(PLACE)
        seq = [(s, 'NNG'), (josa(s, '이', '가'), 'JKS'),
               (p, 'NNG'), (random.choice(JKB_L), 'JKB'),
               ('가', 'VV'), (random.choice(EP_L), 'EP'),
               ('다', 'EF'), ('.', 'SF')]
    else:
        # 유형 D : 짧은 문장                     (예: 나는 갔다)
        n = random.choice(PRON)
        seq = [(n, 'NP'), (josa(n, '은', '는'), 'JX'),
               ('가', 'VV'), ('었', 'EP'), ('다', 'EF'), ('.', 'SF')]
    return seq


pos_data = [make_pos_sentence() for _ in range(2000)]

# 어휘 사전은 반드시 학습 데이터만으로 만들어야 한다 (평가 데이터 정보 누출 방지)
n_train = int(len(pos_data) * 0.8)
pos_train, pos_test = pos_data[:n_train], pos_data[n_train:]

print('전체 문장 수 :', len(pos_data), '| 학습', len(pos_train), '| 평가', len(pos_test))
for s in pos_train[:3]:
    print(s)

전체 문장 수 : 2000 | 학습 1600 | 평가 400
[('친구', 'NNG'), ('가', 'JKS'), ('시장', 'NNG'), ('에', 'JKB'), ('커피', 'NNG'), ('를', 'JKO'), ('읽', 'VV'), ('었', 'EP'), ('다', 'EF'), ('.', 'SF')]
[('동생', 'NNG'), ('이', 'JKS'), ('병원', 'NNG'), ('에서', 'JKB'), ('밥', 'NNG'), ('을', 'JKO'), ('읽', 'VV'), ('었', 'EP'), ('다', 'EF'), ('.', 'SF')]
[('나', 'NP'), ('는', 'JX'), ('가', 'VV'), ('었', 'EP'), ('다', 'EF'), ('.', 'SF')]


### [TODO 2-1] 어휘 사전(vocabulary) 만들기

형태소와 품사 태그를 모두 정수 인덱스로 바꾸어야 신경망의 입력으로 쓸 수 있다.
`0`번은 패딩(PAD), `1`번은 미등록어(UNK)를 위해 비워 둔다.

> **[설명]** 신경망은 글자를 직접 받지 못하므로, 형태소와 태그마다 번호를 붙인 사전을 만들어 숫자로 바꿔야 한다. 0번은 길이를 맞추는 패딩, 1번은 사전에 없는 형태소(UNK)에 남겨 둔다. 사전은 학습 데이터만으로 만들어야 평가 데이터의 정보가 새어 들어가지 않는다.

In [7]:
# (2) (전처리) 어휘 사전 구축

PAD, UNK = 0, 1


def build_vocab(sequences, index, use_unk=True):
    """sequences : [(형태소, 태그), ...] 의 리스트,  index : 0이면 형태소, 1이면 태그"""
    tokens = sorted({pair[index] for seq in sequences for pair in seq})
    vocab = {'<PAD>': PAD, '<UNK>': UNK} if use_unk else {'<PAD>': PAD}
    for t in tokens:
        # 토큰 t 에 다음 인덱스 번호를 부여한다. (힌트 : 현재 사전의 크기)
        # 실습 2 : vocab[t] 에 다음 인덱스를 부여하시오.
        vocab[t] = len(vocab)          # 다음 인덱스 = 현재 사전의 크기
    return vocab


word2idx = build_vocab(pos_train, 0)                  # 학습 데이터만 사용
tag2idx = build_vocab(pos_train, 1, use_unk=False)
idx2tag = {i: t for t, i in tag2idx.items()}
print('형태소 사전 크기 :', len(word2idx), ' 태그 사전 크기 :', len(tag2idx))


형태소 사전 크기 : 45  태그 사전 크기 : 11


In [8]:
# (2-2) (전처리) 정수 인코딩 + 패딩 → 텐서 변환

def encode(dataset, word2idx, tag2idx):
    xs = [[word2idx.get(w, UNK) for w, _ in seq] for seq in dataset]
    ys = [[tag2idx[t] for _, t in seq] for seq in dataset]
    return xs, ys


def pad(seqs, max_len):
    """길이가 서로 다른 문장을 max_len으로 맞춘다. 빈 자리는 0(PAD)으로 채운다."""
    out = np.zeros((len(seqs), max_len), dtype=np.int64)
    for i, s in enumerate(seqs):
        s = s[:max_len]
        out[i, :len(s)] = s
    return out


MAX_LEN = max(len(s) for s in pos_data)

x_tr, y_tr = encode(pos_train, word2idx, tag2idx)
x_te, y_te = encode(pos_test, word2idx, tag2idx)

X_train, Y_train = torch.tensor(pad(x_tr, MAX_LEN)), torch.tensor(pad(y_tr, MAX_LEN))
X_test, Y_test = torch.tensor(pad(x_te, MAX_LEN)), torch.tensor(pad(y_te, MAX_LEN))

print('최대 문장 길이 :', MAX_LEN)
print('학습 데이터    :', X_train.shape, Y_train.shape)
print('평가 데이터    :', X_test.shape, Y_test.shape)
print('\n첫 번째 문장의 정수 인코딩 :', X_train[0].tolist())
print('첫 번째 문장의 태그 인코딩 :', Y_train[0].tolist())

최대 문장 길이 : 10
학습 데이터    : torch.Size([1600, 10]) torch.Size([1600, 10])
평가 데이터    : torch.Size([400, 10]) torch.Size([400, 10])

첫 번째 문장의 정수 인코딩 : [39, 3, 25, 30, 40, 14, 37, 29, 11, 2]
첫 번째 문장의 태그 인코딩 : [7, 5, 7, 3, 7, 4, 10, 2, 1, 9]


> **[결과]** 학습 문장 1,600개만으로 `build_vocab`을 돌려 형태소 사전 45개(PAD·UNK 포함)와 태그 사전 11개(PAD + 태그 10개)를 만들었다.
> - 문장 길이는 최대 10형태소라서 모든 문장을 10칸에 맞추고 빈칸은 0으로 채웠다. 그 결과 학습 데이터는 `(1600, 10)`, 평가 데이터는 `(400, 10)` 크기의 정수 표가 되었다.
> - 첫 문장 '친구가 시장에 커피를 읽었다.'는 `[39, 3, 25, 30, ...]`로 바뀌었다. 생성기가 동사를 무작위로 골라 '커피를 읽었다'처럼 어색한 문장도 섞여 있지만, 품사 패턴을 배우는 데는 문제가 없다.

### [TODO 2-2] Bi-LSTM 태거 정의

강의 그림(임베딩 → Bi-LSTM → 선형 분류기)을 그대로 코드로 옮긴다.

- `nn.Embedding` : 형태소 인덱스를 `emb_dim` 차원 벡터로 변환 (`padding_idx=0`)
- `nn.LSTM(..., bidirectional=True)` : 양방향 문맥 표현 h 생성 → 출력 차원은 `hid_dim * 2`
- `nn.Linear` : 각 시점의 h를 태그 점수로 변환

> **[설명]** 임베딩은 형태소 번호를 뜻이 담긴 벡터로 바꾸고, Bi-LSTM은 그 벡터를 앞뒤 두 방향으로 읽어 형태소마다 문맥 벡터를 만든다. 정방향과 역방향 출력을 이어 붙이므로 크기가 `hid_dim × 2`가 되고, 마지막 선형 층이 이를 태그 수만큼의 점수로 바꾼다. 문장의 모든 시점마다 태그를 하나씩 내는 many-to-many 구조다.

In [9]:
# (3) (모델 측면) Bi-LSTM 태거 정의

class BiLSTMTagger(nn.Module):
    def __init__(self, vocab_size, tag_size, emb_dim=64, hid_dim=64):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        self.lstm = nn.LSTM(emb_dim, hid_dim, batch_first=True, bidirectional=True)
        # 양방향 LSTM 의 출력을 태그 점수로 바꾸는 선형 층
        # (힌트 : 양방향이므로 입력 차원은 hid_dim 의 2배)
        # 실습 3 : self.fc = nn.Linear(?, ?) 를 정의하시오.
        self.fc = nn.Linear(hid_dim * 2, tag_size)

    def forward(self, x):                 # x : (B, L)
        e = self.emb(x)                   # (B, L, emb_dim)
        h, _ = self.lstm(e)               # (B, L, hid_dim*2)
        # 실습 3 : h 를 태그 점수 (B, L, tag_size) 로 변환해 반환하시오.
        return self.fc(h)                 # (B, L, tag_size) — 시점마다 태그 점수


### [TODO 2-3] 학습 루프

`nn.CrossEntropyLoss`는 `(N, C)` 모양의 점수와 `(N,)` 모양의 정답을 받는다.
따라서 `(B, L, C)` 모양의 출력을 `(B*L, C)`로, 정답은 `(B*L,)`로 펼쳐 주어야 한다.
패딩 자리는 손실 계산에서 제외하기 위해 `ignore_index=PAD`를 사용한다.

> **[설명]** 태깅은 시점마다 분류 문제를 하나씩 푸는 것과 같아서, 배치와 시점을 하나로 펼쳐 분류 손실을 한꺼번에 계산한다. 이때 패딩 자리까지 채점하면 '0 뒤에는 0'이라는 의미 없는 패턴을 배우므로 `ignore_index=PAD`로 빼야 한다. 이 절에서는 `flatten_for_loss`로 `(B, L, C)`를 `(B·L, C)`로 펼쳐 10에폭 학습한다.

**TensorDataset · DataLoader**: PyTorch(`torch.utils.data`)의 클래스로, 입력·정답 텐서를 묶어 미니배치 단위로 꺼내 준다.
- `TensorDataset(X, Y)`: 같은 순서의 `X[i]`, `Y[i]`를 한 쌍으로 묶음
- `DataLoader(ds, batch_size=64, shuffle=True)`: 매 에폭마다 순서를 섞어 64개씩 꺼냄

**nn.CrossEntropyLoss**: PyTorch(`torch.nn`)의 손실 클래스로, 소프트맥스와 음의 로그 우도를 한 번에 계산한다.
- 입력: 점수 `(N, C)`, 정답 `(N,)` — 그래서 `(B, L, C)`를 `(B·L, C)`로 펼친다.
- `ignore_index=PAD`: 정답이 0(패딩)인 자리는 손실과 기울기 계산에서 뺀다.

**torch.optim.Adam**: PyTorch의 옵티마이저로, 기울기의 이동 평균으로 파라미터마다 학습률을 조정한다.
- 한 번의 갱신 순서: `optimizer.zero_grad()` → `loss.backward()` → `optimizer.step()`

In [10]:

def flatten_for_loss(logits, yb):
    """(B, L, C) 로짓과 (B, L) 정답을 CrossEntropyLoss 형태로 펼쳐 돌려주시오."""
    # 실습 4 : logits 를 (-1, C) 로, yb 를 (-1,) 로 펼치시오.
    C = logits.size(-1)
    return logits.reshape(-1, C), yb.reshape(-1)

# (4) (학습 과정)

from torch.utils.data import TensorDataset, DataLoader

train_loader = DataLoader(TensorDataset(X_train, Y_train), batch_size=64, shuffle=True)
model = BiLSTMTagger(vocab_size=len(word2idx), tag_size=len(tag2idx)).to(DEVICE)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 10
for epoch in range(1, EPOCHS + 1):
    model.train()
    total = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)                                   # (B, L, C)
        # TODO: logits와 yb를 CrossEntropyLoss가 받을 수 있는 모양으로 펼쳐 손실을 계산하시오.
        # CrossEntropyLoss 는 (N, C) 와 (N,) 을 받는다 — 배치와 시점을 하나로 펼쳐야 한다.
        loss = criterion(*flatten_for_loss(logits, yb))
        loss.backward()
        optimizer.step()
        total += loss.item() * xb.size(0)
    print(f'epoch {epoch:2d}  loss {total / len(train_loader.dataset):.4f}')

epoch  1  loss 1.9713


epoch  2  loss 0.8982


epoch  3  loss 0.2280


epoch  4  loss 0.0598


epoch  5  loss 0.0269


epoch  6  loss 0.0166


epoch  7  loss 0.0117


epoch  8  loss 0.0088


epoch  9  loss 0.0069


epoch 10  loss 0.0056


> **[결과]** `flatten_for_loss`로 로짓과 정답을 펼쳐 Adam으로 10에폭 학습했다. 손실은 1에폭 1.9713에서 4에폭 0.0598로 빠르게 떨어졌고, 10에폭에는 0.0056까지 내려갔다. 문형이 4가지뿐인 생성 데이터라서 몇 에폭 만에 거의 다 외운 셈이다.

**@torch.no_grad()**: PyTorch의 데코레이터로, 함수 안에서는 기울기를 기록하지 않아 평가가 빠르고 메모리를 덜 쓴다.
- `model.eval()`: 드롭아웃 등을 평가 모드로 바꾼다. (이 모델에는 없지만 습관처럼 함께 쓴다)

**argmax와 마스크**
- `logits.argmax(dim=-1)`: 시점마다 점수가 가장 큰 태그 번호를 고른다. → `(B, L)`
- `mask = Y != PAD`: 패딩이 아닌 자리만 `True`. `pred[mask] == Y[mask]`로 실제 형태소 자리만 채점한다.

In [11]:
# (5) (성능 평가) 패딩을 제외한 토큰 단위 정확도

@torch.no_grad()
def token_accuracy(model, X, Y):
    model.eval()
    logits = model(X.to(DEVICE))
    pred = logits.argmax(dim=-1).cpu()
    mask = Y != PAD
    return (pred[mask] == Y[mask]).float().mean().item()


print(f'학습 정확도 : {token_accuracy(model, X_train, Y_train):.4f}')
print(f'평가 정확도 : {token_accuracy(model, X_test, Y_test):.4f}')

학습 정확도 : 1.0000
평가 정확도 : 1.0000


In [12]:
# (5-2) (성능 평가) 중의적인 형태소 '가' 가 문맥에 따라 다르게 태깅되는지 확인

@torch.no_grad()
def predict_pos(model, morphs):
    model.eval()
    ids = torch.tensor([[word2idx.get(m, UNK) for m in morphs]])
    pred = model(ids.to(DEVICE)).argmax(dim=-1)[0].cpu().tolist()
    return list(zip(morphs, [idx2tag[i] for i in pred]))


# 친구가 도서관에서 책을 읽었다  →  '가' 는 주격 조사(JKS)
print(predict_pos(model, ['친구', '가', '도서관', '에서', '책', '을', '읽', '었', '다', '.']))

# 친구가 도서관에 갔다          →  앞의 '가' 는 JKS, 뒤의 '가' 는 동사 어간(VV)
print(predict_pos(model, ['친구', '가', '도서관', '에', '가', '었', '다', '.']))

[('친구', 'NNG'), ('가', 'JKS'), ('도서관', 'NNG'), ('에서', 'JKB'), ('책', 'NNG'), ('을', 'JKO'), ('읽', 'VV'), ('었', 'EP'), ('다', 'EF'), ('.', 'SF')]
[('친구', 'NNG'), ('가', 'JKS'), ('도서관', 'NNG'), ('에', 'JKB'), ('가', 'VV'), ('었', 'EP'), ('다', 'EF'), ('.', 'SF')]


In [13]:
# (5-3) (비교 실험) 단방향 LSTM 이었다면 '가' 를 구분할 수 있었을까?
#   서술형 2-4 의 근거로 추가한 셀. 3부 결과가 바뀌지 않도록 torch 난수 상태를 저장했다가 되돌린다.

class UniLSTMTagger(nn.Module):
    """BiLSTMTagger 와 같고 LSTM 만 단방향(왼쪽 → 오른쪽)인 태거"""
    def __init__(self, vocab_size, tag_size, emb_dim=64, hid_dim=64):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        self.lstm = nn.LSTM(emb_dim, hid_dim, batch_first=True, bidirectional=False)
        self.fc = nn.Linear(hid_dim, tag_size)          # 단방향이므로 hid_dim 그대로

    def forward(self, x):
        h, _ = self.lstm(self.emb(x))
        return self.fc(h)


rng_state = torch.get_rng_state()
torch.manual_seed(SEED)
uni_model = UniLSTMTagger(len(word2idx), len(tag2idx)).to(DEVICE)
uni_opt = torch.optim.Adam(uni_model.parameters(), lr=1e-3)
for epoch in range(EPOCHS):
    uni_model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        uni_opt.zero_grad()
        loss = criterion(*flatten_for_loss(uni_model(xb), yb))
        loss.backward()
        uni_opt.step()
torch.set_rng_state(rng_state)

print(f'단방향 평가 정확도 : {token_accuracy(uni_model, X_test, Y_test):.4f}\n')

# 조사를 생략한 문장 '학교 가었다' — '가' 앞에는 명사만 있고, 뒤의 '었다' 를 봐야 동사임을 알 수 있다.
test = ['학교', '가', '었', '다', '.']
print('양방향 :', predict_pos(model, test))
print('단방향 :', predict_pos(uni_model, test))

단방향 평가 정확도 : 1.0000

양방향 : [('학교', 'NNG'), ('가', 'VV'), ('었', 'EP'), ('다', 'EF'), ('.', 'SF')]
단방향 : [('학교', 'NNG'), ('가', 'JKS'), ('었', 'EP'), ('다', 'EF'), ('.', 'SF')]


> **[결과]** 학습한 양방향 태거는 학습·평가 정확도가 모두 1.0000이었고, 중의적인 형태소 `'가'`도 문맥에 따라 다르게 태깅했다.
> - '친구가 도서관에 갔다'에서 앞의 `가`는 `JKS`(주격 조사), 뒤의 `가`는 `VV`(동사 어간)로 정확히 갈랐다.
> - 같은 구조에서 LSTM만 단방향으로 바꾼 `UniLSTMTagger`도 평가 정확도는 1.0000이었다. 그런데 조사를 생략한 '학교 가었다'를 넣자 양방향은 `가/VV`로 맞혔고, 단방향은 `가/JKS`로 틀렸다. 단방향은 '가' 뒤에 오는 '었다'를 볼 수 없기 때문이다.

### [서술형 2-4]

1. 같은 형태소 `'가'`가 두 문장에서 각각 어떤 품사로 태깅되었는가? 모델은 무엇을 근거로 구분했다고 생각하는가?
2. 단방향 LSTM만 사용했다면 이 구분이 더 어려웠을까? 그 이유를 설명하시오.
3. 학습 정확도와 평가 정확도의 차이를 보고 과적합 여부를 판단하고, 개선 방법을 한 가지 제안하시오.


**서술형 2-4 답안**

**[1] '가'의 품사와 구분 근거** 첫 문장 '친구가 도서관에서 책을 읽었다'의 `가`는 `JKS`로, 두 번째 문장 '친구가 도서관에 갔다'에서는 앞의 `가`가 `JKS`, 뒤의 `가`가 `VV`로 태깅되었다. 모델은 주변 형태소의 품사 패턴을 근거로 삼았다고 생각한다. 명사(`친구`) 바로 뒤에 오는 `가`는 조사이고, 부사격 조사(`에`) 뒤에 오고 바로 다음에 선어말 어미(`었`)가 붙는 `가`는 동사 어간이라는 규칙을 학습 데이터의 유형 A·C에서 배운 것이다. 사람이 '갔다'를 보고 동사라고 아는 것처럼, 모델도 바로 뒤의 `었/EP`와 `다/EF`를 강한 단서로 쓴다.

**[2] 단방향 LSTM이었다면** 이 데이터에서는 큰 차이가 없었지만, 일반 문장에서는 더 어려워진다. 실제로 단방향 태거를 학습해 보니 평가 정확도는 양방향과 같은 1.0000이었다. 생성기의 문장에서는 동사 `가`가 언제나 `에`나 `는` 뒤에 오기 때문에, 왼쪽 문맥만 봐도 구분이 되기 때문이다. 하지만 조사를 생략한 '학교 가었다'를 넣자 단방향은 '명사 뒤의 가는 조사'라는 왼쪽 단서만 믿고 `JKS`로 틀렸고, 양방향은 오른쪽의 `었다`를 보고 `VV`로 맞혔다. 구어체처럼 조사가 자주 빠지는 실제 문장에서는 뒤 문맥이 결정적인 경우가 많으므로, 태깅에는 양방향 구조가 더 적합하다.

**[3] 과적합 여부와 개선 방법** 학습 정확도와 평가 정확도가 모두 1.0000이라 숫자로는 과적합이라고 볼 수 없다. 다만 이 결과를 그대로 믿기는 어렵다. 평가 문장도 같은 생성기에서 같은 어휘 45개와 같은 문형 4가지로 만들어져서, 평가 데이터가 사실상 학습 데이터의 복사본에 가깝기 때문이다. 그래서 이 실험은 과적합을 판단할 힘이 없다. 개선 방법으로는 평가 데이터를 학습에 없던 어휘와 문형(예: 조사 생략, 새로운 동사)으로 따로 만들거나 세종 말뭉치 같은 실제 코퍼스의 일부를 떼어 평가하는 것을 제안한다. 그렇게 해서 성능이 떨어지면 드롭아웃이나 조기 종료로 일반화 성능을 높여야 한다.

---
# 3부. Bi-LSTM 개체명 인식(NER)과 개체 단위 평가

**목적** : 같은 Bi-LSTM 구조를 **개체명 인식**에 적용하고, BIO 태깅 체계와
**개체 단위 F1**으로 성능을 평가한다.

강의에서 살펴본 것처럼 한국어는 조사가 붙기 때문에 어절 단위보다
**음절 단위** 입력이 유리하다. 이번 실습에서는 음절 단위로 문장을 나누고,
각 음절에 BIO 태그를 붙인다.

또한 **평가 데이터에만 등장하는 개체명**(학습 때 본 적 없는 인명·지명·기관명)을 일부러 남겨 두었다.
실제 서비스에서 늘 발생하는 **미등록어(OOV) 문제**를 직접 확인하기 위해서다.

```
이  순  신  은     한  산  도  에  서
B-PER I-PER I-PER O  B-LOC I-LOC I-LOC O  O
```

> **[설명]** 개체명 인식(NER)은 문장에서 사람·장소·기관·날짜 같은 고유한 이름을 찾아 범위와 종류를 표시하는 작업이다. 여러 글자로 된 이름을 표시하려고 시작(B)·안쪽(I)·바깥(O) 태그를 쓰고, 한국어는 이름 뒤에 조사가 붙으므로 음절 단위로 태깅한다. 2부의 Bi-LSTM 태거를 그대로 가져와, 처음 보는 이름 앞에서 모델이 어떻게 흔들리는지 확인한다.

In [14]:
# (1) (데이터 측면) 개체명 인식 학습 데이터 만들기

PER = ['이순신', '세종대왕', '김유신', '유관순', '안중근', '장영실', '신사임당', '정약용']
LOC = ['서울', '부산', '한산도', '제주도', '강릉', '경주', '평양', '독도']
ORG = ['국립국어원', '한국은행', '고려대학교', '삼성전자', '국립중앙박물관', '한국철도공사']
DAT = ['1592년', '2023년 5월', '어제', '지난주', '1919년 3월 1일', '오늘']

# {PER} 등은 개체명 자리, [은/는] 등은 앞말의 받침에 따라 결정되는 조사 자리
TEMPLATES = [
    '{PER}[은/는] {DAT} {LOC}에서 큰 승리를 거두었다.',
    '{DAT} {ORG}[은/는] {LOC}에 새로운 지점을 열었다.',
    '{PER}[은/는] {ORG}에서 연구를 하였다.',
    '{LOC}에 사는 {PER}[은/는] {ORG}에 편지를 보냈다.',
    '{ORG}[은/는] {DAT}부터 {LOC} 지역 조사를 시작한다.',
    '{DAT} {PER}[과/와] {PER}[이/가] {LOC}에서 만났다.',
    '{PER}[의] 일기는 {ORG}[이/가] 보관하고 있다.',
    '{LOC}[과/와] {LOC}[을/를] 잇는 도로가 {DAT} 개통된다.',
]
SLOTS = {'PER': PER, 'LOC': LOC, 'ORG': ORG, 'DAT': DAT}


def make_ner_sentence(slots=SLOTS):
    """(음절 리스트, BIO 태그 리스트) 를 생성한다."""
    tpl = random.choice(TEMPLATES)
    chars, tags, used = [], [], set()
    last = ''
    i = 0
    while i < len(tpl):
        if tpl[i] == '{':                      # 개체명 자리
            j = tpl.index('}', i)
            key = tpl[i + 1:j]
            pool = [v for v in slots[key] if v not in used] or slots[key]
            value = random.choice(pool)
            used.add(value)
            last = value
            for k, ch in enumerate(value):
                chars.append(ch)
                tags.append(('B-' if k == 0 else 'I-') + key)
            i = j + 1
        elif tpl[i] == '[':                    # 조사 자리 (받침에 따라 선택)
            j = tpl.index(']', i)
            forms = tpl[i + 1:j].split('/')
            value = forms[0] if len(forms) == 1 else josa(last, forms[0], forms[1])
            for ch in value:
                chars.append(ch)
                tags.append('O')
            i = j + 1
        else:
            chars.append(tpl[i])
            tags.append('O')
            i += 1
    return chars, tags


# 학습에는 각 유형의 앞쪽 개체만 사용하고, 뒤쪽 2개는 평가 전용(= 미등록 개체명)으로 남겨 둔다.
# 실제 서비스에서 늘 마주치는 '학습 때 본 적 없는 이름' 상황을 재현하기 위한 장치다.
TRAIN_SLOTS = {k: v[:-2] for k, v in SLOTS.items()}
UNSEEN = {k: v[-2:] for k, v in SLOTS.items()}

ner_train_data = [make_ner_sentence(TRAIN_SLOTS) for _ in range(2400)]
ner_test_data = [make_ner_sentence(SLOTS) for _ in range(600)]

print('학습 문장 :', len(ner_train_data), '| 평가 문장 :', len(ner_test_data))
print('평가 전용(미등록) 개체 :', UNSEEN)

demo_chars, demo_tags = ner_train_data[0]
print('\n문장 :', ''.join(demo_chars))
print('태그 :', list(zip(demo_chars, demo_tags))[:14], '...')

학습 문장 : 2400 | 평가 문장 : 600
평가 전용(미등록) 개체 : {'PER': ['신사임당', '정약용'], 'LOC': ['평양', '독도'], 'ORG': ['국립중앙박물관', '한국철도공사'], 'DAT': ['1919년 3월 1일', '오늘']}

문장 : 서울과 부산을 잇는 도로가 어제 개통된다.
태그 : [('서', 'B-LOC'), ('울', 'I-LOC'), ('과', 'O'), (' ', 'O'), ('부', 'B-LOC'), ('산', 'I-LOC'), ('을', 'O'), (' ', 'O'), ('잇', 'O'), ('는', 'O'), (' ', 'O'), ('도', 'O'), ('로', 'O'), ('가', 'O')] ...


### [TODO 3-1] 음절 사전과 태그 사전 만들기

2부와 같은 방식으로 음절 사전(`char2idx`)과 BIO 태그 사전(`ner_tag2idx`)을 만든다.

> **[설명]** 2부와 똑같이 번호 사전을 만들되, 이번에는 형태소 대신 음절 하나하나가 단위다. 음절은 종류가 적어서 처음 보는 단어도 이미 아는 글자로 쪼개질 가능성이 높다. 하지만 학습 데이터에 한 번도 없던 글자는 결국 UNK가 된다.

In [15]:
# (2) (전처리)

char2idx = {'<PAD>': PAD, '<UNK>': UNK}
for cs, _ in ner_train_data:               # 학습 데이터만으로 사전을 만든다
    for c in cs:
        if c not in char2idx:
            # 실습 5 : 새로운 음절 c 에 인덱스를 부여하시오.
            char2idx[c] = len(char2idx)

ner_tag2idx = {'<PAD>': PAD}
for _, tgs in ner_train_data:
    for t in tgs:
        if t not in ner_tag2idx:
            # 실습 5 : 새로운 태그 t 에 인덱스를 부여하시오.
            ner_tag2idx[t] = len(ner_tag2idx)
ner_idx2tag = {i: t for t, i in ner_tag2idx.items()}
print('음절 사전 크기 :', len(char2idx), ' NER 태그 수 :', len(ner_tag2idx))


음절 사전 크기 : 97  NER 태그 수 : 10


In [16]:
# (2-2) 정수 인코딩 + 패딩

NER_MAX_LEN = max(len(c) for c, _ in ner_train_data + ner_test_data)


def encode_ner(dataset):
    xs = pad([[char2idx.get(c, UNK) for c in cs] for cs, _ in dataset], NER_MAX_LEN)
    ys = pad([[ner_tag2idx[t] for t in tg] for _, tg in dataset], NER_MAX_LEN)
    return torch.tensor(xs), torch.tensor(ys)


Xn_train, Yn_train = encode_ner(ner_train_data)
Xn_test, Yn_test = encode_ner(ner_test_data)

unk_ratio = (Xn_test == UNK).sum().item() / (Xn_test != PAD).sum().item()
print('최대 문장 길이 :', NER_MAX_LEN)
print('학습 데이터    :', Xn_train.shape)
print('평가 데이터    :', Xn_test.shape)
print(f'평가 데이터의 미등록 음절(UNK) 비율 : {unk_ratio:.3%}')

최대 문장 길이 : 39
학습 데이터    : torch.Size([2400, 39])
평가 데이터    : torch.Size([600, 39])
평가 데이터의 미등록 음절(UNK) 비율 : 5.213%


> **[결과]** 학습 문장 2,400개로 음절 사전 97개와 BIO 태그 사전 10개(PAD, O, 개체 유형 4가지 × B·I)를 만들었다.
> - 가장 긴 문장이 39음절이라 모든 문장을 39칸에 맞췄다.
> - 평가 데이터의 음절 중 5.213%가 사전에 없는 UNK였다. `정약용`(정·약·용), `오늘`(오·늘), `평양`(평·양)처럼 평가 전용 개체에만 나오는 글자가 UNK로 바뀐 것이다. 학습 데이터에는 UNK가 하나도 없으므로, 모델은 UNK 벡터를 한 번도 학습하지 못했다.

In [17]:
# (3)(4) 모델 정의와 학습 — 2부에서 만든 BiLSTMTagger를 그대로 재사용한다

ner_model = BiLSTMTagger(len(char2idx), len(ner_tag2idx),
                         emb_dim=64, hid_dim=128).to(DEVICE)

ner_loader = DataLoader(TensorDataset(Xn_train, Yn_train), batch_size=64, shuffle=True)
ner_criterion = nn.CrossEntropyLoss(ignore_index=PAD)
ner_optimizer = torch.optim.Adam(ner_model.parameters(), lr=1e-3)

for epoch in range(1, EPOCHS + 1):
    ner_model.train()
    total = 0.0
    for xb, yb in ner_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        ner_optimizer.zero_grad()
        logits = ner_model(xb)
        loss = ner_criterion(logits.reshape(-1, logits.size(-1)), yb.reshape(-1))
        loss.backward()
        ner_optimizer.step()
        total += loss.item() * xb.size(0)
    print(f'epoch {epoch:2d}  loss {total / len(ner_loader.dataset):.4f}')

print(f'\n토큰 단위 정확도 : {token_accuracy(ner_model, Xn_test, Yn_test):.4f}')

epoch  1  loss 1.3112


epoch  2  loss 0.3336


epoch  3  loss 0.0385


epoch  4  loss 0.0122


epoch  5  loss 0.0067


epoch  6  loss 0.0044


epoch  7  loss 0.0032


epoch  8  loss 0.0024


epoch  9  loss 0.0019


epoch 10  loss 0.0015

토큰 단위 정확도 : 0.9553


> **[결과]** 2부의 `BiLSTMTagger`를 `hid_dim=128`로 키워 그대로 다시 썼다. 손실은 10에폭 만에 0.0015까지 떨어졌고, 평가 데이터의 토큰 단위 정확도는 0.9553이었다. 숫자만 보면 높지만, 아래 개체 단위 평가에서 이 숫자가 얼마나 부풀려졌는지 확인한다.

### [TODO 3-2] BIO 태그열에서 개체 추출하기

토큰 단위 정확도는 `O` 태그가 대부분이기 때문에 성능이 과대평가된다.
강의에서 배운 대로 **경계와 유형이 모두 일치한 개체만 정답**으로 인정하는
개체 단위 평가를 하려면, 먼저 BIO 태그열에서 개체를 뽑아내야 한다.

`['B-PER', 'I-PER', 'O', 'B-LOC']` → `{(0, 2, 'PER'), (3, 4, 'LOC')}`
(시작 위치, 끝 위치+1, 개체 유형)

> **[설명]** 토큰 정확도는 이름의 일부만 맞혀도 점수를 주지만, 실제로는 '이순신'을 통째로 찾아야 쓸모가 있다. 그래서 BIO 태그열을 `(시작, 끝, 유형)` 묶음으로 바꾸고, 경계와 유형이 모두 맞은 개체만 정답으로 센다. `extract_entities`는 B를 만나면 새 개체를 열고, O나 다른 유형을 만나면 진행 중이던 개체를 닫는다.

In [18]:
# (5) (성능 평가) 개체 단위 F1

def extract_entities(tag_seq):
    """BIO 태그열 → {(start, end, type), ...}"""
    entities, start, etype = set(), None, None
    for i, tag in enumerate(list(tag_seq) + ['O']):
        if tag.startswith('B-'):
            if etype is not None:
                entities.add((start, i, etype))
            # 실습 6 : 새 개체의 시작 위치와 유형을 기록하시오.
            start, etype = i, tag[2:]
        elif tag.startswith('I-') and etype == tag[2:]:
            continue                       # 같은 개체가 이어지는 중
        else:                              # 'O' 이거나 유형이 바뀐 I- 태그
            if etype is not None:
                # 실습 6 : 진행 중이던 개체를 (시작, 끝, 유형) 으로 저장하시오.
                entities.add((start, i, etype))
            start, etype = None, None
    return entities


# ---- 아래 블록은 위 TODO 를 채운 뒤 정상 동작합니다 ----
try:
    demo = ['B-PER', 'I-PER', 'I-PER', 'O', 'B-LOC', 'I-LOC']
    got = extract_entities(demo)
    print('추출 결과 :', sorted(got))
    assert got == {(0, 3, 'PER'), (4, 6, 'LOC')}, '기대한 개체 집합과 다릅니다.'
    print('통과!')
except NotImplementedError as e:
    print('▶ 미구현 :', e)


추출 결과 : [(0, 3, 'PER'), (4, 6, 'LOC')]
통과!


> **[결과]** `extract_entities`로 `['B-PER', 'I-PER', 'I-PER', 'O', 'B-LOC', 'I-LOC']`에서 `(0, 3, 'PER')`, `(4, 6, 'LOC')` 두 개체를 뽑아 자가 점검을 `통과!`했다. 'O'를 만났을 때 개체를 저장한 뒤 `start, etype`을 비워 두어야 다음 'O'에서 같은 개체가 두 번 저장되지 않는다.

### [TODO 3-3] 정밀도·재현율·F1 계산

$$P=\frac{TP}{TP+FP},\qquad R=\frac{TP}{TP+FN},\qquad F1=\frac{2PR}{P+R}$$

- TP : 정답에도 있고 예측에도 있는 개체 수
- FP : 예측에는 있지만 정답에 없는 개체 수
- FN : 정답에는 있지만 예측하지 못한 개체 수

> **[설명]** 정밀도는 '찾았다고 한 것 중 맞은 비율', 재현율은 '찾아야 할 것 중 찾은 비율'이고, F1은 둘의 조화 평균이다. 개체 수가 적고 O가 대부분인 NER에서는 정확도 대신 이 세 지표로 성능을 본다. `entity_f1`은 개체 집합의 교집합과 차집합으로 TP·FP·FN을 센다.

**집합 연산으로 TP·FP·FN 세기**: 개체를 `(시작, 끝, 유형)` 튜플의 집합으로 두면 파이썬 집합 연산만으로 채점할 수 있다.
- `hyp & gold`(교집합): 경계와 유형이 모두 맞은 개체 → TP
- `hyp - gold`(차집합): 예측했지만 정답에 없는 개체 → FP
- `gold - hyp`(차집합): 정답인데 예측하지 못한 개체 → FN
- 경계가 한 음절만 어긋나도 튜플이 달라지므로 FP 1개와 FN 1개가 동시에 생긴다.

In [19]:
@torch.no_grad()
def entity_f1(model, X, Y, idx2tag_map, verbose=True):
    model.eval()
    pred = model(X.to(DEVICE)).argmax(dim=-1).cpu()
    tp = fp = fn = 0
    for i in range(len(X)):
        length = int((Y[i] != PAD).sum())
        gold_tags = [idx2tag_map[int(t)] for t in Y[i][:length]]
        pred_tags = [idx2tag_map[int(t)] for t in pred[i][:length]]
        gold, hyp = extract_entities(gold_tags), extract_entities(pred_tags)
        # 실습 7 : 집합 연산으로 TP·FP·FN 을 누적하고 P·R·F1 을 구하시오.
        tp += len(hyp & gold)          # 맞게 찾은 개체 (경계·유형 모두 일치)
        fp += len(hyp - gold)          # 잘못 찾은 개체
        fn += len(gold - hyp)          # 놓친 개체
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    if verbose:
        print(f'개체 단위  정밀도 {p:.4f}  재현율 {r:.4f}  F1 {f1:.4f}   (TP {tp} / FP {fp} / FN {fn})')
    return p, r, f1


In [20]:
# (5-1) 평가 데이터 전체에 대해 토큰 단위 정확도와 개체 단위 F1 을 비교한다.
#   원본 노트북에는 entity_f1 을 호출하는 셀이 없어, 서술형 3-4 의 근거로 쓰기 위해 추가했다.

tok_acc = token_accuracy(ner_model, Xn_test, Yn_test)
print(f'토큰 단위 정확도 : {tok_acc:.4f}')
p, r, f1 = entity_f1(ner_model, Xn_test, Yn_test, ner_idx2tag)

# O 태그가 차지하는 비율 — 모든 음절을 O 로 찍어도 얻는 '공짜 정확도'
o_ratio = (Yn_test == ner_tag2idx['O']).sum().item() / (Yn_test != PAD).sum().item()
print(f'\n평가 데이터에서 O 태그 비율 : {o_ratio:.4f}  (전부 O 로 예측해도 이만큼은 맞는다)')

토큰 단위 정확도 : 0.9553


개체 단위  정밀도 0.7632  재현율 0.8361  F1 0.7980   (TP 1428 / FP 443 / FN 280)

평가 데이터에서 O 태그 비율 : 0.5886  (전부 O 로 예측해도 이만큼은 맞는다)


In [21]:
# (5-1-2) 미등록 개체명(UNSEEN)이 들어 있는 문장과 그렇지 않은 문장을 나누어 평가한다.

unseen_names = {v for vs in UNSEEN.values() for v in vs}
has_unseen = torch.tensor([any(n in ''.join(cs) for n in unseen_names) for cs, _ in ner_test_data])
print(f'미등록 개체 포함 문장 : {int(has_unseen.sum())}개 / 미포함 문장 : {int((~has_unseen).sum())}개\n')

print('[미등록 개체 미포함 문장]')
entity_f1(ner_model, Xn_test[~has_unseen], Yn_test[~has_unseen], ner_idx2tag)
print('\n[미등록 개체 포함 문장]')
entity_f1(ner_model, Xn_test[has_unseen], Yn_test[has_unseen], ner_idx2tag)

# 미등록 개체 하나하나를 맞혔는지 유형별로 집계
@torch.no_grad()
def unseen_entity_recall(model):
    model.eval()
    pred = model(Xn_test.to(DEVICE)).argmax(dim=-1).cpu()
    stat = {}
    for i, (cs, tg) in enumerate(ner_test_data):
        sent = ''.join(cs)
        gold = extract_entities(tg)
        hyp = extract_entities([ner_idx2tag[int(t)] for t in pred[i][:len(cs)]])
        for (s, e, typ) in gold:
            if sent[s:e] in unseen_names:
                hit, tot = stat.get(sent[s:e], (0, 0))
                stat[sent[s:e]] = (hit + ((s, e, typ) in hyp), tot + 1)
    return stat

print('\n[미등록 개체별 재현율]')
for name, (hit, tot) in sorted(unseen_entity_recall(ner_model).items(), key=lambda x: x[1][0] / x[1][1]):
    typ = [k for k, v in UNSEEN.items() if name in v][0]
    print(f'  {typ}  {name:<12s} {hit:3d} / {tot:3d}  ({hit / tot:.2f})')

미등록 개체 포함 문장 : 364개 / 미포함 문장 : 236개

[미등록 개체 미포함 문장]


개체 단위  정밀도 1.0000  재현율 1.0000  F1 1.0000   (TP 663 / FP 0 / FN 0)

[미등록 개체 포함 문장]


개체 단위  정밀도 0.6333  재현율 0.7321  F1 0.6791   (TP 765 / FP 443 / FN 280)

[미등록 개체별 재현율]


  PER  신사임당           0 /  54  (0.00)
  DAT  1919년 3월 1일    0 /  78  (0.00)
  ORG  한국철도공사         0 /  66  (0.00)
  DAT  오늘             4 /  56  (0.07)
  LOC  평양            39 /  57  (0.68)
  PER  정약용           26 /  38  (0.68)
  LOC  독도            56 /  56  (1.00)
  ORG  국립중앙박물관       70 /  70  (1.00)


> **[결과]** `entity_f1`로 평가 문장 600개를 채점했다.
> - 토큰 정확도는 0.9553이지만 개체 단위 F1은 0.7980(정밀도 0.7632, 재현율 0.8361)에 그쳤다. 평가 데이터의 58.86%가 O 태그라서, 전부 O로 찍어도 정확도는 0.59가 나온다.
> - 미등록 개체가 없는 문장 236개는 F1 1.0000으로 하나도 틀리지 않았고, 미등록 개체가 있는 문장 364개는 F1 0.6791이었다. FP 443개와 FN 280개가 모두 미등록 개체가 있는 문장에서 나왔다.
> - 미등록 개체별로 보면 `신사임당`(0/54), `1919년 3월 1일`(0/78), `한국철도공사`(0/66)는 한 번도 맞히지 못했고, `독도`(56/56)와 `국립중앙박물관`(70/70)은 모두 맞혔다.

In [22]:
# (5-2) 새로운 문장에 대한 개체명 인식 결과 확인

@torch.no_grad()
def predict_ner(model, sentence):
    model.eval()
    chars = list(sentence)
    ids = torch.tensor([[char2idx.get(c, UNK) for c in chars]]).to(DEVICE)
    pred = model(ids).argmax(dim=-1)[0].cpu().tolist()
    tags = [ner_idx2tag[i] for i in pred]
    ents = extract_entities(tags)
    print('문장 :', sentence)
    for s, e, t in sorted(ents):
        print(f'  {sentence[s:e]:<12s} → {t}')
    return list(zip(chars, tags))


# (a) 학습 데이터에 있던 개체명
predict_ner(ner_model, '유관순은 지난주 서울에서 큰 승리를 거두었다.')

# (b) 학습 데이터에 없던 개체명 (UNSEEN) — 미등록어 문제 확인
predict_ner(ner_model, '정약용은 오늘 독도에서 큰 승리를 거두었다.')
predict_ner(ner_model, '한국철도공사는 오늘부터 독도 지역 조사를 시작한다.')

문장 : 유관순은 지난주 서울에서 큰 승리를 거두었다.
  유관순          → PER
  지난주          → DAT
  서울           → LOC
문장 : 정약용은 오늘 독도에서 큰 승리를 거두었다.
  정            → PER
  오            → DAT
  독도           → LOC
문장 : 한국철도공사는 오늘부터 독도 지역 조사를 시작한다.
  한            → ORG
  국철도공사        → ORG
  독도           → LOC


[('한', 'B-ORG'),
 ('국', 'B-ORG'),
 ('철', 'I-ORG'),
 ('도', 'I-ORG'),
 ('공', 'I-ORG'),
 ('사', 'I-ORG'),
 ('는', 'O'),
 (' ', 'O'),
 ('오', 'O'),
 ('늘', 'O'),
 ('부', 'O'),
 ('터', 'O'),
 (' ', 'O'),
 ('독', 'B-LOC'),
 ('도', 'I-LOC'),
 (' ', 'O'),
 ('지', 'O'),
 ('역', 'O'),
 (' ', 'O'),
 ('조', 'O'),
 ('사', 'O'),
 ('를', 'O'),
 (' ', 'O'),
 ('시', 'O'),
 ('작', 'O'),
 ('한', 'O'),
 ('다', 'O'),
 ('.', 'O')]

In [23]:
# (5-4) BIO 규칙을 어긴 예측 집계 — 서술형 3-4 (CRF) 의 근거로 추가한 셀
#   · I-X 는 반드시 B-X 나 I-X 뒤에만 와야 한다.  (예: O → I-DAT 는 불가능)
#   · 학습 데이터에는 개체 두 개가 띄어쓰기 없이 붙는 경우가 없으므로 B-X → B-Y 도 한 번도 나오지 않는다.
from collections import Counter

ner_pred = ner_model(Xn_test.to(DEVICE)).argmax(dim=-1).cpu()
bad_I, bad_BB = Counter(), Counter()
for i, (cs, _) in enumerate(ner_test_data):
    tags = ['O'] + [ner_idx2tag[int(t)] for t in ner_pred[i][:len(cs)]]
    for a, b in zip(tags, tags[1:]):
        if b.startswith('I-') and (a == 'O' or a[2:] != b[2:]):
            bad_I[(a, b)] += 1
        if a.startswith('B-') and b.startswith('B-'):
            bad_BB[(a, b)] += 1

print(f'규칙을 어긴 I- 태그 : {sum(bad_I.values())}개', bad_I.most_common(3))
print(f'B- 바로 뒤에 B- 가 온 경우 : {sum(bad_BB.values())}개', bad_BB.most_common(3))

규칙을 어긴 I- 태그 : 139개 [(('O', 'I-DAT'), 101), (('I-LOC', 'I-ORG'), 18), (('B-LOC', 'I-ORG'), 10)]
B- 바로 뒤에 B- 가 온 경우 : 110개 [(('B-ORG', 'B-ORG'), 47), (('B-PER', 'B-PER'), 43), (('B-ORG', 'B-PER'), 10)]


> **[결과]** `predict_ner`로 새 문장을 넣어 보았다.
> - 학습 때 본 `유관순`, `지난주`, `서울`은 셋 다 정확히 찾았다.
> - 처음 보는 `정약용`은 `정`만 PER로, `오늘`은 `오`만 DAT로 잡아 경계가 잘렸다. `한국철도공사`는 `한/B-ORG, 국/B-ORG, ...`로 태깅되어 `한`과 `국철도공사` 두 기관으로 쪼개졌다.
> - 평가 데이터 전체에서 BIO 규칙을 어긴 I- 태그가 139개였고, 그중 101개가 `O → I-DAT`였다. `B-ORG → B-ORG`(47개), `B-PER → B-PER`(43개)처럼 학습 데이터에 없던 B 연속도 나왔다. 모델이 태그를 음절마다 따로 고르기 때문에 앞뒤 태그가 맞물리는지 확인하지 않는다는 뜻이다.

### [서술형 3-4]

1. 토큰 단위 정확도와 개체 단위 F1의 값 차이를 비교하고, 왜 그런 차이가 나는지 설명하시오.
2. 평가 데이터에만 있는 개체명(`UNSEEN`)이 포함된 문장에서 모델이 어떤 오류를 냈는지 직접 확인하고,
   미등록어(OOV) 문제와 그 완화 방법을 설명하시오.
3. 강의에서 배운 CRF 층을 이 모델에 추가한다면 어떤 오류를 줄일 수 있겠는가? 구체적인 예를 들어 설명하시오.


**서술형 3-4 답안**

**[1] 토큰 정확도와 개체 F1의 차이** 같은 모델인데도 토큰 단위 정확도는 0.9553, 개체 단위 F1은 0.7980으로 약 16%p 차이가 났다. 첫째, 평가 데이터의 58.86%가 O 태그라서 모델이 개체를 하나도 못 찾아도 정확도 절반 이상은 공짜로 얻는다. 둘째, 토큰 정확도는 부분 점수를 주지만 개체 F1은 주지 않는다. 평가 데이터에서 6번 나온 것처럼 `정약용`을 `정/B-PER, 약/I-PER, 용/O`로 태깅하면, 토큰 정확도에서는 3개 중 2개를 맞힌 것으로 친다. 하지만 개체 단위에서는 점수를 하나도 얻지 못하고, `정약`이라는 틀린 개체(FP)와 놓친 `정약용`(FN)이 하나씩 늘어난다. 개체명 인식의 목적이 이름을 통째로 찾는 데 있으므로, 실제 성능은 개체 단위 F1로 판단해야 한다.

**[2] 미등록 개체에서 생긴 오류와 완화 방법** 오류는 미등록 개체에서만 나왔다. 미등록 개체가 없는 문장은 F1 1.0000이었지만, 있는 문장은 0.6791로 떨어졌다. 흥미로운 점은 글자를 처음 보는지와 맞히는지가 일치하지 않았다는 것이다. `1919년 3월 1일`은 모든 글자가 사전에 있는데도 78번 중 한 번도 맞히지 못했다. 가장 흔한 예측은 `1/B, 9/I, 1/B, 9/I, 년/I`처럼 두 번째 '1'에서 개체를 새로 시작하고 띄어쓰기를 O로 끊는 것이었다. 학습 때 숫자 '1'은 '1592년'의 첫 글자로만 나왔으니, 모델이 '1이면 날짜의 시작'이라고 외운 셈이다. `한국철도공사`가 `한`과 `국철도공사`로 쪼개진 것도 같은 이유다. '국'은 학습 데이터에서 '국립국어원'의 첫 글자로 B-ORG가 376번 붙었다. 반대로 `정약용`은 세 글자가 모두 UNK인데도 38번 중 26번을 맞혔다. '…은 오늘 …에서 큰 승리를 거두었다' 같은 문장 틀에서 이름이 들어갈 자리를 문맥으로 알아챈 것이다. 하지만 UNK가 이어지면 판단이 흔들려서, `오늘`(오·늘 모두 UNK)은 `O → I-DAT`처럼 규칙에도 맞지 않는 태그를 내며 56번 중 4번만 맞혔다. 학습 데이터에는 UNK가 하나도 없어서 UNK 벡터가 한 번도 학습되지 않았기 때문이다. 정리하면 이 모델은 이름을 일반화했다기보다, 글자별로 외운 위치 습관과 문장 틀에 크게 기대고 있다.

완화 방법으로는 먼저, 학습할 때 음절 일부를 일부러 UNK로 바꿔(word dropout) 모델이 UNK 자리에서도 주변 문맥으로 태그를 고르도록 훈련할 수 있다. 템플릿의 개체 자리에 여러 이름을 바꿔 넣어 데이터를 늘리는 방법, 자모나 서브워드처럼 더 작은 단위를 함께 쓰는 방법도 있다. 가장 확실한 방법은 KLUE-BERT처럼 큰 말뭉치로 사전학습한 모델을 미세조정해, 이 실습에서 처음 보는 이름도 이미 아는 표현으로 시작하게 하는 것이다.

**[3] CRF 층을 추가한다면** CRF는 태그와 태그 사이의 전이 점수를 학습해, 문장 전체에서 가장 그럴듯한 태그열을 한 번에 고른다. 지금 모델은 음절마다 태그를 따로 고르기 때문에 BIO 규칙을 어긴 I- 태그가 139개나 나왔다. CRF는 학습 데이터에 한 번도 없던 `O → I-DAT` 전이(101개)와 `I-LOC → I-ORG` 전이(18개)에 매우 낮은 점수를 주므로, 이런 태그열은 거의 고르지 않는다. 예를 들어 `신사임당`을 `B-PER, B-PER, I-PER, I-PER`로 태깅한 경우(54번 중 30번), CRF는 학습 데이터에서 본 적 없는 `B-PER → B-PER`보다 `B-PER → I-PER`를 훨씬 높게 쳐서 `B-PER, I-PER, I-PER, I-PER`를 고를 가능성이 높다. 같은 이유로 `한국철도공사`의 `B-ORG → B-ORG`(47개)도 하나의 기관으로 이어 붙일 수 있다. 다만 CRF는 태그의 앞뒤를 맞출 뿐, UNK 글자가 무엇인지 알려 주지는 못한다. 그래서 `오늘`처럼 글자 정보가 아예 없는 경우까지 해결하려면 [2]의 OOV 대책을 함께 써야 한다.

---
## (심화 · 선택) 더 해 볼 것

여유가 있다면 아래 중 하나를 시도하고 결과를 간단히 정리해 보자. (가산점)

1. **CRF 층 추가** : `torchcrf` (`pip install pytorch-crf`)의 `CRF` 모듈을 Bi-LSTM 위에 얹어
   개체 단위 F1이 얼마나 향상되는지 비교한다.
2. **패딩 처리 개선** : 이번 실습에서는 손실 계산에서만 패딩을 제외했지만,
   `nn.utils.rnn.pack_padded_sequence`를 사용하면 역방향 LSTM이 패딩을 아예 읽지 않게 만들 수 있다.
   적용 전후의 성능을 비교해 보자.
3. **사전학습 모델 미세조정** : Hugging Face `transformers`의
   `AutoModelForTokenClassification`과 `klue/bert-base`를 이용해 같은 데이터를 학습시키고
   Bi-LSTM 결과와 비교한다.
4. **실제 말뭉치 적용** : KLUE-NER 등 공개 데이터셋을 내려받아 같은 파이프라인을 적용해 본다.

## 제출 안내

- 파일명 : `13주차_실습_학번_이름.ipynb`
- 모든 셀을 순서대로 실행하여 **출력 결과가 남아 있는 상태**로 저장할 것
- 서술형 답안은 노트북 안의 답안 작성란에 직접 작성할 것

---
# [심화] 심화 과제 풀이

위 '더 해 볼 것' 네 가지를 모두 해 보았다.

| 번호 | 과제 | 바꾼 것 | 데이터 |
|---|---|---|---|
| 심화 1 | CRF 층 추가 | 출력층: 음절별 argmax → CRF + 비터비 디코딩 (학습만 / BIO 제약) | 3부 NER 데이터 |
| 심화 2 | 패딩 처리 개선 | LSTM 입력: 패딩 포함 → `pack_padded_sequence` | 3부 NER 데이터 |
| 심화 3 | 사전학습 모델 미세조정 | 모델 전체: Bi-LSTM → `klue/bert-base` | 3부 NER 데이터 |
| 심화 4 | 실제 말뭉치 적용 | 데이터: 템플릿 생성 문장 → KLUE-NER | KLUE-NER |

심화 1·2는 시드 하나로 비교하면 우연에 좌우되므로, 기본 Bi-LSTM까지 시드 3개(13·14·15)로 다시 학습해 평균을 비교한다.
BERT는 CPU에서 학습 시간이 길어 시드 13 한 번만 학습한다.

> **[설명]** 3부의 Bi-LSTM은 처음 보는 이름 앞에서 경계를 자주 잘랐고(미등록 개체 포함 문장 F1 0.6791), BIO 규칙을 어긴 태그도 139개 냈다. 심화 과제 네 가지는 이 약점을 서로 다른 쪽에서 고친다. CRF는 태그끼리의 연결을, 패킹은 패딩이 끼어드는 문제를, BERT는 사전학습한 언어 지식을, KLUE-NER는 실제 문장에서 버티는 힘을 확인한다. 모든 모델을 같은 음절 단위 BIO 채점 함수로 비교하려고 공통 도구(`summarize`)를 먼저 만들었다.

In [24]:
# (심화 공통) 여러 모델을 같은 기준으로 비교하기 위한 도구
#   · 모든 모델의 예측을 '음절별 태그 문자열 리스트'로 바꾼 뒤 같은 함수로 채점한다.
#   · 미등록 개체 포함/미포함 문장의 F1, BIO 규칙 위반 수, 미등록 개체별 재현율을 함께 본다.
import time
import pandas as pd

SEEDS = [13, 14, 15]
test_gold = [tg for _, tg in ner_test_data]
test_len = [len(cs) for cs, _ in ner_test_data]
seen_idx = [i for i in range(len(ner_test_data)) if not has_unseen[i]]
unseen_idx = [i for i in range(len(ner_test_data)) if has_unseen[i]]


def train_tagger(model, loader, epochs=EPOCHS, lr=1e-3, log_every=0):
    """CrossEntropy 로 학습한다. model 에 loss() 가 있으면(CRF) 그것을 쓴다. 학습 시간(초)을 돌려준다."""
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    ce = nn.CrossEntropyLoss(ignore_index=PAD)
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        total, n = 0.0, 0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            if hasattr(model, 'loss'):
                loss = model.loss(xb, yb)
            else:
                logits = model(xb)
                loss = ce(logits.reshape(-1, logits.size(-1)), yb.reshape(-1))
            loss.backward()
            opt.step()
            total += loss.item() * xb.size(0)
            n += xb.size(0)
        if log_every and (epoch % log_every == 0 or epoch == epochs):
            print(f'  epoch {epoch:2d}  loss {total / n:.4f}')
    return time.time() - t0


@torch.no_grad()
def lstm_tags(model, X, lengths, idx2tag_map=None):
    """Bi-LSTM 계열 모델의 예측을 음절별 태그 리스트로 바꾼다."""
    idx2tag_map = idx2tag_map or ner_idx2tag
    model.eval()
    pred = model(X.to(DEVICE)).argmax(dim=-1).cpu()
    return [[idx2tag_map.get(int(t), 'O') if int(t) != PAD else 'O' for t in pred[i][:n]]
            for i, n in enumerate(lengths)]


def prf(gold_lists, pred_lists, etype=None):
    """개체 단위 정밀도·재현율·F1. etype 을 주면 그 유형만 채점한다."""
    tp = fp = fn = 0
    for g, h in zip(gold_lists, pred_lists):
        G, H = extract_entities(g), extract_entities(h)
        if etype:
            G = {e for e in G if e[2] == etype}
            H = {e for e in H if e[2] == etype}
        tp += len(G & H); fp += len(H - G); fn += len(G - H)
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return p, r, (2 * p * r / (p + r) if p + r else 0.0)


def bio_errors(pred_lists):
    """(규칙을 어긴 I- 태그 수, B- 바로 뒤 B- 수)"""
    bad_i = bad_bb = 0
    for tags in pred_lists:
        prev = 'O'
        for t in tags:
            if t.startswith('I-') and (prev == 'O' or prev[2:] != t[2:]):
                bad_i += 1
            if prev.startswith('B-') and t.startswith('B-'):
                bad_bb += 1
            prev = t
    return bad_i, bad_bb


def unseen_recall(pred_lists):
    """미등록 개체 이름별 재현율 {이름: 맞힌 비율}"""
    stat = {}
    for (cs, tg), h in zip(ner_test_data, pred_lists):
        sent, H = ''.join(cs), extract_entities(h)
        for (s, e, typ) in extract_entities(tg):
            if sent[s:e] in unseen_names:
                hit, tot = stat.get(sent[s:e], (0, 0))
                stat[sent[s:e]] = (hit + ((s, e, typ) in H), tot + 1)
    return {k: hit / tot for k, (hit, tot) in stat.items()}


def summarize(name, pred_lists, seed=None, sec=None):
    acc = np.mean([t == g for P, G in zip(pred_lists, test_gold) for t, g in zip(P, G)])
    p, r, f1 = prf(test_gold, pred_lists)
    f1_seen = prf([test_gold[i] for i in seen_idx], [pred_lists[i] for i in seen_idx])[2]
    f1_unseen = prf([test_gold[i] for i in unseen_idx], [pred_lists[i] for i in unseen_idx])[2]
    bi, bb = bio_errors(pred_lists)
    return {'모델': name, '시드': seed, '토큰 정확도': acc, '정밀도': p, '재현율': r, '개체 F1': f1,
            'F1(미등록 없음)': f1_seen, 'F1(미등록 포함)': f1_unseen,
            'I- 규칙 위반': bi, 'B→B': bb, '학습 시간(초)': sec}


rows, recalls, best = [], {}, {}       # 시드별 결과 / 미등록 개체 재현율 / 시드 13 모델


def run_variant(name, make_model, lr=1e-3):
    for seed in SEEDS:
        torch.manual_seed(seed)
        m = make_model().to(DEVICE)
        sec = train_tagger(m, ner_loader, lr=lr)
        tags = lstm_tags(m, Xn_test, test_len)
        rows.append(summarize(name, tags, seed, sec))
        if seed == SEEDS[0]:
            best[name], recalls[name] = m, unseen_recall(tags)
        print(f'{name:<22s} seed {seed}  개체 F1 {rows[-1]["개체 F1"]:.4f}  ({sec:.0f}초)')


# 위 3부에서 학습한 ner_model 의 점수도 같은 함수로 다시 채점해 맞는지 확인한다.
chk = summarize('3부 ner_model', lstm_tags(ner_model, Xn_test, test_len))
print(f"3부 ner_model 다시 채점 : 개체 F1 {chk['개체 F1']:.4f}  (위 entity_f1 결과와 같아야 한다)")

3부 ner_model 다시 채점 : 개체 F1 0.7980  (위 entity_f1 결과와 같아야 한다)


> **[설명]** 비교의 기준선이다. 3부의 `ner_model`은 앞의 실습에서 난수를 쓴 뒤에 만들어져, 같은 시드로 새 모델을 만들어도 똑같이 재현되지 않는다. 그래서 기본 Bi-LSTM도 심화 모델과 똑같은 절차(시드 고정 → 모델 생성 → 10에폭 학습)로 다시 학습해, 바꾼 부분 하나의 효과만 비교한다.

In [25]:
# (심화 기준선) 기본 Bi-LSTM 을 시드 3개로 다시 학습한다. (3부와 같은 구조 : emb 64, hidden 128, 10 에폭)
run_variant('Bi-LSTM (기본)', lambda: BiLSTMTagger(len(char2idx), len(ner_tag2idx), emb_dim=64, hid_dim=128))

Bi-LSTM (기본)           seed 13  개체 F1 0.7617  (33초)


Bi-LSTM (기본)           seed 14  개체 F1 0.7696  (21초)


Bi-LSTM (기본)           seed 15  개체 F1 0.7803  (16초)


## 심화 1. CRF 층 추가 (`pytorch-crf`)

> **[설명]** CRF(조건부 무작위장)는 태그와 태그 사이의 '연결 점수'를 배워서, 음절마다 따로 고르는 대신 문장 전체에서 가장 그럴듯한 태그열을 한 번에 고르는 층이다. 'O 다음에 I-DAT는 오지 않는다', 'B-PER 다음에는 I-PER가 온다' 같은 규칙을 데이터에서 스스로 익히므로, 3부에서 본 `O → I-DAT`(101개), `B-PER → B-PER`(43개) 같은 오류를 줄이려고 쓴다. 이 실습에서는 `pytorch-crf`의 `CRF`를 `BiLSTMTagger` 위에 얹은 `BiLSTMCRF`로 구현했다. 전이 점수를 데이터로만 배우는 CRF와, BIO 규칙상 불가능한 전이(`I-X` 앞에 `B-X`·`I-X`가 아닌 태그)를 처음부터 -10000으로 막은 'BIO 제약' CRF를 함께 비교한다.

In [26]:
# 심화 1 설치 — 최초 1회만 실행
%pip install -q pytorch-crf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


**CRF**: pytorch-crf(`torchcrf`)의 클래스로, 선형 사슬 CRF 층이다. 태그 수 × 태그 수 크기의 전이 점수 행렬을 학습한다.
- `CRF(num_tags, batch_first=True)`: `transitions[i, j]` = 태그 i 다음에 태그 j가 올 점수 (`start_transitions`, `end_transitions`도 있음)
- `crf(emissions, tags, mask, reduction='mean')`: 정답 태그열의 **로그 우도**를 돌려준다. 손실로 쓰려면 부호를 바꾼다.
- `crf.decode(emissions, mask)`: 비터비 알고리즘으로 점수가 가장 높은 태그열을 `list[list[int]]`로 돌려준다.
- `mask`: 실제 음절 자리만 `True`. 첫 시점은 반드시 `True`여야 한다.

**CRF 점수 읽는 법**
- 문장 점수 = Σ(음절마다 emission 점수) + Σ(이웃한 두 태그의 전이 점수)
- 전이 점수가 크게 음수이면 그 연결은 비터비 디코딩에서 거의 뽑히지 않는다.

In [27]:
# 심화 1 : Bi-LSTM 위에 CRF 층을 얹는다.
#   · Bi-LSTM 은 음절마다 '이 태그일 점수'(emission) 만 내고,
#   · CRF 는 '앞 태그 → 뒤 태그' 전이 점수(transitions) 를 더해 문장 전체에서 가장 점수가 높은 태그열을 고른다.
from torchcrf import CRF


class BiLSTMCRF(nn.Module):
    def __init__(self, vocab_size, tag_size, emb_dim=64, hid_dim=128, constrain=False):
        super().__init__()
        self.encoder = BiLSTMTagger(vocab_size, tag_size, emb_dim, hid_dim)   # emission 점수 (B, L, C)
        self.crf = CRF(tag_size, batch_first=True)                           # 전이 점수 (C, C)
        self.tag_size = tag_size
        if constrain:
            # BIO 제약 : I-X 는 B-X 나 I-X 뒤에만 올 수 있다. 그 밖의 전이와 '문장 첫 태그가 I-X' 를 -10000 으로 막는다.
            #   (학습률 1e-3 의 Adam 은 한 번에 0.001 정도만 움직이므로 학습 중에도 사실상 그대로 남는다)
            with torch.no_grad():
                for a_tag, i in ner_tag2idx.items():
                    for b_tag, j in ner_tag2idx.items():
                        if b_tag.startswith('I-') and a_tag[2:] != b_tag[2:]:
                            self.crf.transitions[i, j] = -1e4
                for b_tag, j in ner_tag2idx.items():
                    if b_tag.startswith('I-'):
                        self.crf.start_transitions[j] = -1e4

    def loss(self, x, y):
        # crf(...) 는 정답 태그열의 로그 우도를 돌려주므로 부호를 바꿔 손실로 쓴다. 패딩은 mask 로 뺀다.
        return -self.crf(self.encoder(x), y, mask=(x != PAD), reduction='mean')

    def forward(self, x):
        # 비터비 디코딩으로 고른 태그열을 원-핫으로 돌려준다. → 기존 평가 함수(argmax)를 그대로 쓸 수 있다.
        paths = self.crf.decode(self.encoder(x), mask=(x != PAD))
        out = torch.zeros(x.size(0), x.size(1), self.tag_size, device=x.device)
        for i, path in enumerate(paths):
            out[i, torch.arange(len(path)), torch.tensor(path)] = 1.0
        return out


run_variant('Bi-LSTM + CRF', lambda: BiLSTMCRF(len(char2idx), len(ner_tag2idx), emb_dim=64, hid_dim=128))
run_variant('Bi-LSTM + CRF(BIO 제약)', lambda: BiLSTMCRF(len(char2idx), len(ner_tag2idx), emb_dim=64, hid_dim=128, constrain=True))

Bi-LSTM + CRF          seed 13  개체 F1 0.7665  (35초)


Bi-LSTM + CRF          seed 14  개체 F1 0.7714  (32초)


Bi-LSTM + CRF          seed 15  개체 F1 0.7841  (36초)


Bi-LSTM + CRF(BIO 제약)  seed 13  개체 F1 0.8982  (36초)


Bi-LSTM + CRF(BIO 제약)  seed 14  개체 F1 0.8097  (34초)


Bi-LSTM + CRF(BIO 제약)  seed 15  개체 F1 0.8641  (36초)


In [28]:
# 심화 1-2 : CRF 가 배운 전이 점수 확인 — 학습 데이터에 없던 전이는 점수가 낮아야 한다.
T = best['Bi-LSTM + CRF'].crf.transitions.detach().cpu()
t2i = ner_tag2idx
print('[학습만 한 CRF 의 전이 점수]  (BIO 제약 CRF 는 막힌 전이가 -10000)')
pairs = [('O', 'B-DAT'), ('O', 'I-DAT'),
         ('B-PER', 'I-PER'), ('B-PER', 'B-PER'),
         ('B-ORG', 'I-ORG'), ('B-ORG', 'B-ORG'),
         ('I-LOC', 'I-LOC'), ('I-LOC', 'I-ORG')]
for a, b in pairs:
    print(f'  {a:>6s} → {b:<6s} : {T[t2i[a], t2i[b]]:+.2f}')

# 3부에서 틀렸던 문장을 CRF 모델로 다시 태깅해 본다.
for sent in ['정약용은 오늘 독도에서 큰 승리를 거두었다.', '한국철도공사는 오늘부터 독도 지역 조사를 시작한다.']:
    for name in ['Bi-LSTM (기본)', 'Bi-LSTM + CRF', 'Bi-LSTM + CRF(BIO 제약)']:
        print(f'[{name}]', end=' ')
        _ = predict_ner(best[name], sent)

[학습만 한 CRF 의 전이 점수]  (BIO 제약 CRF 는 막힌 전이가 -10000)
       O → B-DAT  : +0.15
       O → I-DAT  : -0.08
   B-PER → I-PER  : +0.08
   B-PER → B-PER  : -0.08
   B-ORG → I-ORG  : +0.08
   B-ORG → B-ORG  : -0.00
   I-LOC → I-LOC  : +0.09
   I-LOC → I-ORG  : +0.04
[Bi-LSTM (기본)] 문장 : 정약용은 오늘 독도에서 큰 승리를 거두었다.
  정약용          → PER
[Bi-LSTM + CRF] 문장 : 정약용은 오늘 독도에서 큰 승리를 거두었다.
  정약용          → PER
[Bi-LSTM + CRF(BIO 제약)] 문장 : 정약용은 오늘 독도에서 큰 승리를 거두었다.
  정약용          → PER
  독도           → LOC
[Bi-LSTM (기본)] 문장 : 한국철도공사는 오늘부터 독도 지역 조사를 시작한다.
  한            → ORG
  국철도공         → ORG
[Bi-LSTM + CRF] 문장 : 한국철도공사는 오늘부터 독도 지역 조사를 시작한다.
  한            → ORG
  국철도공         → ORG
[Bi-LSTM + CRF(BIO 제약)] 문장 : 한국철도공사는 오늘부터 독도 지역 조사를 시작한다.
  한국철도공사       → ORG
  독도           → LOC


> **[결과]** CRF를 두 가지로 얹어 보았다.
> - 전이 점수를 데이터로만 배운 CRF는 10에폭 뒤에도 전이 점수가 ±0.15 안쪽이었다(`O → I-DAT` −0.08, `B-PER → B-PER` −0.08). 학습 데이터에서는 Bi-LSTM 점수만으로 이미 정답 태그열이 나와서, 전이 점수를 바꿀 기울기가 거의 생기지 않았기 때문이다. 그래서 `한국철도공사`를 기본 모델과 똑같이 `한`과 `국철도공`으로 쪼갰다.
> - BIO 규칙상 불가능한 전이를 −10000으로 막은 CRF는 같은 문장에서 `한국철도공사`를 기관 하나로 묶었고, 다른 두 모델이 놓친 `독도`도 찾았다.

## 심화 2. 패딩 처리 개선 (`pack_padded_sequence`)

> **[설명]** 패딩은 길이를 맞추려고 넣은 빈칸인데, 문장 뒤에 붙어 있어서 역방향 LSTM은 빈칸 수십 개를 먼저 읽은 뒤에야 실제 음절에 도착한다. 손실에서 패딩을 빼더라도, 은닉 상태에는 이미 빈칸을 읽은 흔적이 남는다. `pack_padded_sequence`는 문장마다 실제 길이까지만 LSTM을 돌려 이 흔적을 없앤다. 이 실습에서는 `BiLSTMTagger`의 `forward`만 바꾼 `PackedBiLSTMTagger`로 구현하고, 같은 문장에 패딩 길이만 바꿔 넣어 출력이 달라지는지도 확인했다.

**pack_padded_sequence · pad_packed_sequence**: PyTorch(`torch.nn.utils.rnn`)의 함수로, 길이가 다른 문장 묶음을 RNN이 실제 길이까지만 계산하게 만든다.
- `pack_padded_sequence(e, lengths, batch_first=True, enforce_sorted=False)`: 패딩을 뺀 `PackedSequence`로 묶는다. `lengths`는 CPU 텐서여야 하고, `enforce_sorted=False`면 길이순으로 미리 정렬하지 않아도 된다.
- `nn.LSTM`은 `PackedSequence`를 그대로 받는다. 역방향 LSTM도 문장마다 실제 마지막 음절부터 읽는다.
- `pad_packed_sequence(h, batch_first=True, total_length=L)`: 다시 `(B, L, hidden)` 텐서로 펼친다. 패딩 자리는 0으로 채워진다.

In [29]:
# 심화 2 : 역방향 LSTM 이 패딩을 읽지 않도록 pack_padded_sequence 를 쓴다.
#   패딩이 문장 '뒤'에 붙어 있으므로, 기본 모델의 역방향 LSTM 은 패딩(0) 수십 칸을 먼저 읽고 나서 실제 음절에 도착한다.

class PackedBiLSTMTagger(BiLSTMTagger):
    def forward(self, x):                                   # x : (B, L)
        lengths = (x != PAD).sum(dim=1).cpu()               # 문장마다 실제 길이
        e = self.emb(x)
        packed = nn.utils.rnn.pack_padded_sequence(e, lengths, batch_first=True, enforce_sorted=False)
        h, _ = self.lstm(packed)                            # 실제 길이까지만 계산
        h, _ = nn.utils.rnn.pad_packed_sequence(h, batch_first=True, total_length=x.size(1))
        return self.fc(h)                                   # 패딩 자리는 0 벡터 → 손실에서 어차피 빠진다


run_variant('Bi-LSTM + pack', lambda: PackedBiLSTMTagger(len(char2idx), len(ner_tag2idx), emb_dim=64, hid_dim=128))

Bi-LSTM + pack         seed 13  개체 F1 0.7563  (53초)


Bi-LSTM + pack         seed 14  개체 F1 0.7635  (42초)


Bi-LSTM + pack         seed 15  개체 F1 0.7778  (40초)


In [30]:
# 심화 2-2 : 같은 문장에 패딩 길이만 바꿔 넣었을 때 출력이 달라지는가?
#   패딩이 결과에 영향을 주면 안 되는데, 기본 모델은 역방향 LSTM 때문에 영향을 받는다.
sent = '정약용은 오늘 독도에서 큰 승리를 거두었다.'
ids = [char2idx.get(c, UNK) for c in sent]
x_nopad = torch.tensor([ids])
x_pad = torch.tensor([ids + [PAD] * (NER_MAX_LEN - len(ids))])

with torch.no_grad():
    for name in ['Bi-LSTM (기본)', 'Bi-LSTM + pack']:
        m = best[name].eval()
        a, b = m(x_nopad)[0], m(x_pad)[0, :len(ids)]
        diff_tags = int((a.argmax(-1) != b.argmax(-1)).sum())
        print(f'{name:<22s} 로짓 최대 차이 {float((a - b).abs().max()):.4f}   태그가 바뀐 음절 {diff_tags}개')

Bi-LSTM (기본)           로짓 최대 차이 3.5223   태그가 바뀐 음절 0개
Bi-LSTM + pack         로짓 최대 차이 0.0000   태그가 바뀐 음절 0개


> **[결과]** 같은 문장에 패딩 길이만 바꿔 넣자 기본 모델은 로짓이 최대 3.5223까지 달라졌고, pack 모델은 0.0000으로 똑같았다. `pack_padded_sequence`가 역방향 LSTM이 패딩을 읽는 문제를 실제로 없앤 것이다. 다만 기본 모델도 태그가 바뀐 음절은 0개였다. 학습 때부터 늘 문장 뒤에 패딩이 붙은 채로 배워서, 패딩을 읽은 흔적이 남아도 태그 순위까지 뒤집히지는 않았다.

In [31]:
# 심화 1·2 비교 : 시드 3개의 평균 (± 표준편차)
df = pd.DataFrame(rows)
cols = ['개체 F1', 'F1(미등록 없음)', 'F1(미등록 포함)', 'I- 규칙 위반', 'B→B', '학습 시간(초)']
agg = df.groupby('모델', sort=False)[cols].agg(['mean', 'std'])
show = pd.DataFrame({c: [f'{m:.4f} ± {s:.4f}' if c.startswith(('개체', 'F1')) else f'{m:.0f} ± {s:.0f}'
                         for m, s in zip(agg[(c, 'mean')], agg[(c, 'std')].fillna(0))] for c in cols},
                    index=agg.index)
display(show)

print('\n[시드별 개체 F1]')
display(df.pivot(index='모델', columns='시드', values='개체 F1').loc[list(dict.fromkeys(df['모델']))].round(4))

print('\n[미등록 개체별 재현율 — 시드 13]')
display(pd.DataFrame(recalls).round(2))

,개체 F1,F1(미등록 없음),F1(미등록 포함),I- 규칙 위반,B→B,학습 시간(초)
모델,,,,,,
Bi-LSTM (기본),0.7705 ± 0.0093,1.0000 ± 0.0000,0.6149 ± 0.0113,288 ± 82,40 ± 45,23 ± 8
Bi-LSTM + CRF,0.7740 ± 0.0091,1.0000 ± 0.0000,0.6212 ± 0.0108,267 ± 81,32 ± 44,34 ± 2
Bi-LSTM + CRF(BIO 제약),0.8574 ± 0.0446,1.0000 ± 0.0000,0.7618 ± 0.0743,0 ± 0,0 ± 0,35 ± 1
Bi-LSTM + pack,0.7658 ± 0.0109,1.0000 ± 0.0000,0.6069 ± 0.0126,300 ± 58,38 ± 37,45 ± 7



[시드별 개체 F1]


시드,13,14,15
모델,,,
Bi-LSTM (기본),0.7617,0.7696,0.7803
Bi-LSTM + CRF,0.7665,0.7714,0.7841
Bi-LSTM + CRF(BIO 제약),0.8982,0.8097,0.8641
Bi-LSTM + pack,0.7563,0.7635,0.7778



[미등록 개체별 재현율 — 시드 13]


,Bi-LSTM (기본),Bi-LSTM + CRF,Bi-LSTM + CRF(BIO 제약),Bi-LSTM + pack
독도,0.00,0.00,1.00,0.00
평양,0.00,0.00,1.00,0.00
신사임당,0.00,0.15,0.87,0.00
1919년 3월 1일,0.00,0.00,0.00,0.00
정약용,0.82,0.87,1.00,0.87
한국철도공사,0.00,0.00,1.00,0.00
국립중앙박물관,0.00,0.00,0.17,0.00
오늘,0.00,0.00,0.00,0.00


> **[결과]** 시드 3개 평균 개체 F1은 기본 0.7705, 학습만 한 CRF 0.7740, BIO 제약 CRF 0.8574, pack 0.7658이었다.
> - 차이를 만든 것은 BIO 제약 하나였다. 규칙을 어긴 I- 태그가 평균 288개에서 0개가 되었고, 미등록 개체가 든 문장의 F1이 0.6149에서 0.7618로 올랐다. 시드 13에서는 `한국철도공사`·`독도`·`평양`의 재현율이 0에서 1.00으로 바뀌었다.
> - 대신 BIO 제약 CRF는 시드별 F1이 0.8097~0.8982로 표준편차가 0.0446이나 되어, 다른 모델(0.009~0.011)보다 훨씬 많이 흔들렸다. `1919년 3월 1일`과 `오늘`은 제약을 걸어도 하나도 못 찾았다.
> - 학습만 한 CRF(+0.0035)와 pack(−0.0047)의 차이는 시드 간 표준편차(약 0.01) 안쪽이라 효과가 없다고 보는 것이 맞다. pack은 CPU에서 평균 학습 시간이 23초에서 45초로 오히려 늘었다.
> - 기본 모델은 같은 구조인데도 다시 학습하니 0.7617~0.7803으로, 3부의 `ner_model`(0.7980)보다 낮았다. 3부에서 56/56이던 `독도`도 시드 13 재학습 모델은 하나도 못 찾았다. 미등록 개체 결과는 시드에 크게 흔들린다.

## 심화 3. 사전학습 모델 미세조정 (`klue/bert-base`)

> **[설명]** 사전학습 모델 미세조정은 위키백과·뉴스 같은 큰 말뭉치로 언어를 미리 배운 모델(BERT)에 분류층 하나만 얹고, 내 데이터로 조금 더 학습시키는 방법이다. 모델이 이미 수많은 이름과 문장 구조를 알고 있어서, 학습 데이터에 없던 '정약용'이나 '한국철도공사'도 처음 보는 단어가 아닐 수 있다. 이 실습에서는 `klue/bert-base`를 `AutoModelForTokenClassification`으로 불러와 3부 데이터로 3에폭 미세조정했고, 토큰 예측을 음절로 펼쳐 Bi-LSTM과 같은 기준으로 채점했다.

**AutoTokenizer · AutoModelForTokenClassification**: Hugging Face `transformers`의 클래스로, 모델 이름만으로 맞는 토크나이저와 '토큰마다 라벨 하나'를 내는 모델을 불러온다.
- `AutoTokenizer.from_pretrained('klue/bert-base')`: KLUE용 WordPiece 토크나이저. `'정약용은'` → `['정약용', '##은']` (`##`은 앞 토큰에 붙는 조각)
- `tokenizer(문장들, padding=True, truncation=True, return_offsets_mapping=True)`: `input_ids`, `attention_mask`와 함께 토큰마다 원문의 `(시작, 끝)` 음절 위치를 돌려준다. `[CLS]`, `[SEP]`, 패딩은 `(0, 0)`.
- `AutoModelForTokenClassification.from_pretrained(이름, num_labels=…)`: BERT 위에 토큰 분류층(`Linear(768, 라벨 수)`)을 새로 얹는다. 분류층만 무작위로 초기화된다.
- `model(input_ids, attention_mask, labels)`: `labels`를 주면 `.loss`(교차 엔트로피, `-100` 자리는 제외)와 `.logits` `(B, T, 라벨 수)`를 돌려준다.

**get_linear_schedule_with_warmup**: `transformers`의 함수로, 학습률을 처음 10% 동안 0에서 올렸다가 끝까지 0으로 선형으로 줄인다. 미세조정에서 사전학습 가중치를 처음부터 크게 흔들지 않으려고 쓴다.

In [32]:
# 심화 3 : klue/bert-base 를 같은 NER 데이터로 미세조정한다.
#   BERT 는 음절이 아니라 WordPiece 토큰 단위로 입력을 받는다. ('정약용은' → '정약용', '##은')
#   → 학습할 때는 토큰마다 '첫 음절의 태그'를 정답으로 주고,
#   → 평가할 때는 토큰의 예측을 다시 음절로 펼쳐, 위의 Bi-LSTM 과 똑같이 음절 단위 BIO 로 채점한다.
import transformers
from transformers import AutoTokenizer, AutoModelForTokenClassification

transformers.logging.set_verbosity_error()
BERT_NAME = 'klue/bert-base'
tokenizer = AutoTokenizer.from_pretrained(BERT_NAME)


def bert_encode(dataset):
    sents = [''.join(cs) for cs, _ in dataset]
    enc = tokenizer(sents, padding=True, truncation=True, max_length=64,
                    return_offsets_mapping=True, return_tensors='pt')
    labels = torch.full(enc['input_ids'].shape, -100)          # -100 : 손실에서 빼는 자리 ([CLS], [SEP], 패딩)
    for i, (cs, tg) in enumerate(dataset):
        for j, (s, e) in enumerate(enc['offset_mapping'][i].tolist()):
            if e > s:
                labels[i, j] = ner_tag2idx[tg[s]]                # 토큰의 첫 음절 태그
    return enc, labels


def bert_to_char_tags(pred_ids, offsets, sent):
    """토큰 예측 → 음절 태그. 토큰의 첫 음절은 예측 태그, 나머지 음절은 같은 유형의 I- 로 채운다."""
    tags = ['O'] * len(sent)
    for p, (s, e) in zip(pred_ids, offsets):
        if e <= s:
            continue
        t = ner_idx2tag.get(p, 'O') if p != PAD else 'O'
        tags[s] = t
        for k in range(s + 1, min(e, len(sent))):
            tags[k] = 'O' if t == 'O' else 'I-' + t[2:]
    # 띄어쓰기는 토큰에 들어가지 않는다. 바로 뒤 음절이 I- 면 같은 개체 안의 공백('2023년 5월')으로 본다.
    for k in range(len(sent) - 1):
        if sent[k] == ' ' and tags[k + 1].startswith('I-'):
            tags[k] = tags[k + 1]
    return tags


bert_train = ner_train_data[:len(ner_train_data)]
tr_enc, tr_lab = bert_encode(bert_train)
te_enc, te_lab = bert_encode(ner_test_data)
print('학습 입력 :', tuple(tr_enc['input_ids'].shape), '| 평가 입력 :', tuple(te_enc['input_ids'].shape))
print('토큰화 예 :', tokenizer.convert_ids_to_tokens(tr_enc['input_ids'][0][:16]))
print('토큰 정답 :', [ner_idx2tag.get(int(t), '-') if t != -100 else '-' for t in tr_lab[0][:16]])

학습 입력 : (2400, 21) | 평가 입력 : (600, 25)
토큰화 예 : ['[CLS]', '서울', '##과', '부산', '##을', '잇', '##는', '도로', '##가', '어제', '개통', '##된', '##다', '.', '[SEP]', '[PAD]']
토큰 정답 : ['-', 'B-LOC', 'O', 'B-LOC', 'O', 'O', 'O', 'O', 'O', 'B-DAT', 'O', 'O', 'O', 'O', '-', '-']


In [33]:
# 심화 3-2 : 미세조정 (AdamW, lr 5e-5, 배치 32, 학습률 선형 감소)
from transformers import get_linear_schedule_with_warmup

BERT_EPOCHS = 3
torch.manual_seed(SEED)
bert = AutoModelForTokenClassification.from_pretrained(
    BERT_NAME, num_labels=len(ner_tag2idx), id2label=ner_idx2tag, label2id=ner_tag2idx).to(DEVICE)
print(f'BERT 파라미터 수 : {sum(p.numel() for p in bert.parameters()):,}')
print(f'Bi-LSTM 파라미터 수 : {sum(p.numel() for p in ner_model.parameters()):,}')

bert_loader = DataLoader(TensorDataset(tr_enc['input_ids'], tr_enc['attention_mask'], tr_lab),
                         batch_size=32, shuffle=True)
bert_opt = torch.optim.AdamW(bert.parameters(), lr=5e-5, weight_decay=0.01)
total_steps = BERT_EPOCHS * len(bert_loader)
sched = get_linear_schedule_with_warmup(bert_opt, int(0.1 * total_steps), total_steps)

t0 = time.time()
for epoch in range(1, BERT_EPOCHS + 1):
    bert.train()
    total = 0.0
    for ids, att, lab in bert_loader:
        bert_opt.zero_grad()
        out = bert(input_ids=ids.to(DEVICE), attention_mask=att.to(DEVICE), labels=lab.to(DEVICE))
        out.loss.backward()
        bert_opt.step()
        sched.step()
        total += out.loss.item() * ids.size(0)
    print(f'epoch {epoch}  loss {total / len(bert_train):.4f}   ({time.time() - t0:.0f}초)')
bert_sec = time.time() - t0

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BERT 파라미터 수 : 110,034,442
Bi-LSTM 파라미터 수 : 207,434


epoch 1  loss 0.3896   (198초)


epoch 2  loss 0.0025   (415초)


epoch 3  loss 0.0018   (652초)


In [34]:
# 심화 3-3 : 평가 — 토큰 예측을 음절로 펼쳐 같은 기준으로 채점한다.
@torch.no_grad()
def bert_predict_tags(enc, sents, bs=64):
    bert.eval()
    out = []
    for k in range(0, len(sents), bs):
        logits = bert(input_ids=enc['input_ids'][k:k + bs].to(DEVICE),
                      attention_mask=enc['attention_mask'][k:k + bs].to(DEVICE)).logits
        pred = logits.argmax(-1).cpu().tolist()
        for j, p in enumerate(pred):
            out.append(bert_to_char_tags(p, enc['offset_mapping'][k + j].tolist(), sents[k + j]))
    return out


test_sents = [''.join(cs) for cs, _ in ner_test_data]
bert_tags = bert_predict_tags(te_enc, test_sents)
rows.append(summarize('klue/bert-base', bert_tags, SEED, bert_sec))
recalls['klue/bert-base'] = unseen_recall(bert_tags)
display(pd.DataFrame([rows[-1]]).drop(columns=['시드']).round(4))

for sent in ['정약용은 오늘 독도에서 큰 승리를 거두었다.', '한국철도공사는 오늘부터 독도 지역 조사를 시작한다.',
             '신사임당은 1919년 3월 1일 평양에서 큰 승리를 거두었다.']:
    enc = tokenizer([sent], return_offsets_mapping=True, return_tensors='pt')
    tags = bert_predict_tags(enc, [sent])[0]
    print('문장 :', sent, ' | 토큰 :', tokenizer.tokenize(sent)[:6], '...')
    for s, e, t in sorted(extract_entities(tags)):
        print(f'  {sent[s:e]:<14s} → {t}')

,모델,토큰 정확도,정밀도,재현율,개체 F1,F1(미등록 없음),F1(미등록 포함),I- 규칙 위반,B→B,학습 시간(초)
0,klue/bert-base,0.9994,0.9953,0.9953,0.9953,1.0,0.9923,13,0,652.3949


문장 : 정약용은 오늘 독도에서 큰 승리를 거두었다.  | 토큰 : ['정약용', '##은', '오늘', '독도', '##에서', '큰'] ...
  정약용            → PER
  오늘             → DAT
  독도             → LOC
문장 : 한국철도공사는 오늘부터 독도 지역 조사를 시작한다.  | 토큰 : ['한국', '##철도', '##공사', '##는', '오늘', '##부터'] ...
  한국철도공사         → ORG
  오늘             → DAT
  독도             → LOC
문장 : 신사임당은 1919년 3월 1일 평양에서 큰 승리를 거두었다.  | 토큰 : ['신사', '##임', '##당', '##은', '1919', '##년'] ...
  신사             → PER
  1919년 3월 1일    → DAT
  평양             → LOC


In [35]:
# 심화 1·2·3 종합 : 3부 NER 데이터에서 네 모델 비교 (Bi-LSTM 계열은 시드 3개 평균, BERT 는 시드 13)
df = pd.DataFrame(rows)
summary = df.groupby('모델', sort=False)[['토큰 정확도', '개체 F1', 'F1(미등록 없음)', 'F1(미등록 포함)',
                                         'I- 규칙 위반', 'B→B', '학습 시간(초)']].mean()
display(summary.round(4))
print('\n[미등록 개체별 재현율 — 시드 13]')
display(pd.DataFrame(recalls).round(2))

,토큰 정확도,개체 F1,F1(미등록 없음),F1(미등록 포함),I- 규칙 위반,B→B,학습 시간(초)
모델,,,,,,,
Bi-LSTM (기본),0.9335,0.7705,1.0,0.6149,288.3333,39.6667,23.2115
Bi-LSTM + CRF,0.9335,0.7740,1.0,0.6212,267.0000,32.0000,34.4885
Bi-LSTM + CRF(BIO 제약),0.9378,0.8574,1.0,0.7618,0.0000,0.0000,35.3719
Bi-LSTM + pack,0.9343,0.7658,1.0,0.6069,300.3333,38.0000,44.9555
klue/bert-base,0.9994,0.9953,1.0,0.9923,13.0000,0.0000,652.3949



[미등록 개체별 재현율 — 시드 13]


,Bi-LSTM (기본),Bi-LSTM + CRF,Bi-LSTM + CRF(BIO 제약),Bi-LSTM + pack,klue/bert-base
독도,0.00,0.00,1.00,0.00,1.00
평양,0.00,0.00,1.00,0.00,1.00
신사임당,0.00,0.15,0.87,0.00,0.85
1919년 3월 1일,0.00,0.00,0.00,0.00,1.00
정약용,0.82,0.87,1.00,0.87,1.00
한국철도공사,0.00,0.00,1.00,0.00,1.00
국립중앙박물관,0.00,0.00,0.17,0.00,1.00
오늘,0.00,0.00,0.00,0.00,1.00


> **[결과]** `klue/bert-base`를 3에폭(652초) 미세조정하자 개체 F1 0.9953, 미등록 개체가 든 문장의 F1 0.9923이 나왔다. Bi-LSTM 계열 최고인 BIO 제약 CRF(0.8574)보다 0.14 높다.
> - 미등록 개체 8개 중 7개를 100% 찾았다. Bi-LSTM이 한 번도 맞히지 못한 `1919년 3월 1일`과 `오늘`도 모두 맞혔다. 토크나이저가 '정약용', '오늘', '독도', '1919'를 이미 토큰 하나로 갖고 있을 만큼, 사전학습 때 이 단어들을 많이 보았기 때문이다.
> - `신사임당`만 0.85로 덜 맞았다. 토크나이저가 '신사 / ##임 / ##당'으로 나눠, 흔한 단어 '신사'에서 개체를 끊는 경우가 있었다. 위 예문에서도 '신사'까지만 PER로 잡았다.
> - 대가는 크다. 파라미터가 110,034,442개로 Bi-LSTM(207,434개)의 약 530배이고, CPU 학습 시간은 652초로 Bi-LSTM 평균(23초)의 약 28배였다.

## 심화 4. 실제 말뭉치 적용 (KLUE-NER)

> **[설명]** KLUE-NER는 한국어 이해 평가 묶음(KLUE)의 개체명 인식 데이터로, 뉴스·위키 문장 2만여 개에 인명(PS)·지명(LC)·기관(OG)·날짜(DT)·시간(TI)·수량(QT) 6가지 개체가 음절 단위 BIO로 붙어 있다. 템플릿 8개로 만든 3부 데이터와 달리 문장 구조와 이름이 실제로 다양하므로, 같은 Bi-LSTM이 진짜 문장에서 얼마나 버티는지 확인하려고 쓴다. 이 실습에서는 `datasets.load_dataset('klue/klue', 'ner')`로 불러와 3부와 같은 `BiLSTMTagger`를 학습하고 개체 유형별 F1을 구했다.

**load_dataset**: Hugging Face `datasets`의 함수로, 공개 데이터셋을 내려받아 `DatasetDict`(train / validation …)로 돌려준다. 한 번 받으면 캐시에 저장된다.
- `load_dataset('klue/klue', 'ner')`: KLUE 묶음 중 NER 설정. 각 예시는 `sentence`, `tokens`(음절 리스트), `ner_tags`(태그 번호 리스트)
- `ds['train'].features['ner_tags'].feature.names`: 태그 번호 → 이름 (`'B-DT'`, …, `'O'`)
- `ds['train'].select(range(n))`: 앞에서 n개만 골라 쓴다.

**collate_fn**: `DataLoader`가 예시 여러 개를 배치 하나로 묶는 함수다. 여기서는 배치 안에서 가장 긴 문장 길이로만 패딩해(`pad_collate`), 최대 145음절에 모두 맞출 때보다 계산을 줄인다.

In [36]:
# 심화 4 설치 — 최초 1회만 실행 (Hugging Face 데이터셋 라이브러리)
%pip install -q datasets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [37]:
# 심화 4 : KLUE-NER 불러오기 — 이미 음절 단위 BIO 태그라서 3부 파이프라인에 그대로 넣을 수 있다.
from collections import Counter
from datasets import load_dataset

klue = load_dataset('klue/klue', 'ner')
klue_names = klue['train'].features['ner_tags'].feature.names
print(klue)
print('태그 :', klue_names)

k_train = [(ex['tokens'], [klue_names[t] for t in ex['ner_tags']]) for ex in klue['train'].select(range(len(klue['train'])))]
k_val = [(ex['tokens'], [klue_names[t] for t in ex['ner_tags']]) for ex in klue['validation']]

ex_cs, ex_tg = k_train[0]
print('\n예문 :', ''.join(ex_cs))
print('개체 :', [(''.join(ex_cs[s:e]), t) for s, e, t in sorted(extract_entities(ex_tg))])

lens = [len(cs) for cs, _ in k_train]
print(f'\n학습 {len(k_train)}문장 / 검증 {len(k_val)}문장 | 문장 길이 평균 {np.mean(lens):.1f}, 최대 {max(lens)}음절')
cnt = Counter(t for _, tg in k_train for (_, _, t) in extract_entities(tg))
print('학습 데이터 개체 수 :', dict(cnt.most_common()))

DatasetDict({
    train: Dataset({
        features: ['sentence', 'tokens', 'ner_tags'],
        num_rows: 21008
    })
    validation: Dataset({
        features: ['sentence', 'tokens', 'ner_tags'],
        num_rows: 5000
    })
})
태그 : ['B-DT', 'I-DT', 'B-LC', 'I-LC', 'B-OG', 'I-OG', 'B-PS', 'I-PS', 'B-QT', 'I-QT', 'B-TI', 'I-TI', 'O']



예문 : 특히 영동고속도로 강릉 방향 문막휴게소에서 만종분기점까지 5㎞ 구간에는 승용차 전용 임시 갓길차로제를 운영하기로 했다.
개체 : [('영동고속도로', 'LC'), ('강릉', 'LC'), ('문막휴게소', 'LC'), ('만종분기점', 'LC'), ('5㎞', 'QT')]

학습 21008문장 / 검증 5000문장 | 문장 길이 평균 54.8, 최대 145음절


학습 데이터 개체 수 : {'PS': 14453, 'QT': 11717, 'OG': 8491, 'DT': 8029, 'LC': 6663, 'TI': 2020}


In [38]:
# 심화 4-2 : 사전 구축 + 문장 길이가 제각각이므로 배치마다 그 배치의 최대 길이로만 패딩한다 (collate_fn)
k_char2idx = {'<PAD>': PAD, '<UNK>': UNK}
for cs, _ in k_train:
    for c in cs:
        k_char2idx.setdefault(c, len(k_char2idx))
k_tag2idx = {'<PAD>': PAD}
for t in klue_names:
    k_tag2idx[t] = len(k_tag2idx)
k_idx2tag = {i: t for t, i in k_tag2idx.items()}


def k_encode(data):
    return [([k_char2idx.get(c, UNK) for c in cs], [k_tag2idx[t] for t in tg]) for cs, tg in data]


def pad_collate(batch):
    L = max(len(x) for x, _ in batch)
    X = torch.zeros(len(batch), L, dtype=torch.long)
    Y = torch.zeros(len(batch), L, dtype=torch.long)
    for i, (x, y) in enumerate(batch):
        X[i, :len(x)] = torch.tensor(x)
        Y[i, :len(y)] = torch.tensor(y)
    return X, Y


k_train_enc, k_val_enc = k_encode(k_train), k_encode(k_val)
val_unk = sum(x.count(UNK) for x, _ in k_val_enc) / sum(len(x) for x, _ in k_val_enc)
print('음절 사전 크기 :', len(k_char2idx), '| 태그 수 :', len(k_tag2idx))
print(f'검증 데이터의 미등록 음절(UNK) 비율 : {val_unk:.3%}')

음절 사전 크기 : 1967 | 태그 수 : 14
검증 데이터의 미등록 음절(UNK) 비율 : 0.064%


In [39]:
# 심화 4-3 : 3부와 같은 Bi-LSTM (emb 64, hidden 128) 을 KLUE-NER 로 학습한다.
KLUE_EPOCHS = 5
torch.manual_seed(SEED)
k_loader = DataLoader(k_train_enc, batch_size=64, shuffle=True, collate_fn=pad_collate)
k_model = BiLSTMTagger(len(k_char2idx), len(k_tag2idx), emb_dim=64, hid_dim=128).to(DEVICE)
k_sec = train_tagger(k_model, k_loader, epochs=KLUE_EPOCHS, log_every=1)
print(f'학습 시간 : {k_sec:.0f}초')

  epoch  1  loss 0.5749


  epoch  2  loss 0.2551


  epoch  3  loss 0.2009


  epoch  4  loss 0.1707


  epoch  5  loss 0.1504
학습 시간 : 325초


In [40]:
# 심화 4-4 : 검증 데이터 5,000문장 평가 — 전체와 개체 유형별 F1
@torch.no_grad()
def k_predict(model, data_enc, bs=256):
    model.eval()
    out = []
    for k in range(0, len(data_enc), bs):
        X, _ = pad_collate(data_enc[k:k + bs])
        pred = model(X.to(DEVICE)).argmax(-1).cpu()
        for j, (x, _) in enumerate(data_enc[k:k + bs]):
            out.append([k_idx2tag.get(int(t), 'O') if int(t) != PAD else 'O' for t in pred[j][:len(x)]])
    return out


k_gold = [tg for _, tg in k_val]
k_pred = k_predict(k_model, k_val_enc)
k_acc = np.mean([p == g for P, G in zip(k_pred, k_gold) for p, g in zip(P, G)])
o_share = np.mean([g == 'O' for G in k_gold for g in G])
print(f'토큰 정확도 {k_acc:.4f}  (O 태그 비율 {o_share:.4f})')

TYPE_NAME = {'PS': '인명', 'LC': '지명', 'OG': '기관', 'DT': '날짜', 'TI': '시간', 'QT': '수량'}
val_cnt = Counter(t for G in k_gold for (_, _, t) in extract_entities(G))
k_rows = []
for t in ['PS', 'LC', 'OG', 'DT', 'TI', 'QT', None]:
    p, r, f1 = prf(k_gold, k_pred, t)
    k_rows.append({'유형': f'{t} ({TYPE_NAME[t]})' if t else '전체 (micro)', '검증 개체 수': val_cnt[t] if t else sum(val_cnt.values()),
                   '정밀도': p, '재현율': r, 'F1': f1})
k_df = pd.DataFrame(k_rows).set_index('유형')
display(k_df.round(4))
print(f"유형별 F1 단순 평균 (macro) : {k_df['F1'].iloc[:-1].mean():.4f}")
print('BIO 규칙 위반 (I-, B→B) :', bio_errors(k_pred))

토큰 정확도 0.9350  (O 태그 비율 0.8138)


,검증 개체 수,정밀도,재현율,F1
유형,,,,
PS (인명),4418,0.6603,0.5002,0.5692
LC (지명),1649,0.5571,0.3845,0.4550
OG (기관),2182,0.5613,0.4409,0.4938
DT (날짜),2312,0.7960,0.7578,0.7764
TI (시간),545,0.8238,0.8752,0.8488
QT (수량),3151,0.8150,0.8067,0.8108
전체 (micro),14257,0.7090,0.6016,0.6509


유형별 F1 단순 평균 (macro) : 0.6590
BIO 규칙 위반 (I-, B→B) : (3096, 49)


In [41]:
# 심화 4-5 : 예측 결과 확인 — KLUE 검증 문장과, 3부의 템플릿 문장
def show_k(sent, gold=None):
    x = [k_char2idx.get(c, UNK) for c in sent]
    tags = k_predict(k_model, [(x, [0] * len(x))])[0]
    print('문장 :', sent)
    print('  예측 :', [(sent[s:e], t) for s, e, t in sorted(extract_entities(tags))])
    if gold:
        print('  정답 :', [(sent[s:e], t) for s, e, t in sorted(extract_entities(gold))])


for i in [0, 1, 2]:
    show_k(''.join(k_val[i][0]), k_val[i][1])
show_k('정약용은 오늘 독도에서 큰 승리를 거두었다.')
show_k('한국철도공사는 오늘부터 독도 지역 조사를 시작한다.')

문장 : 경찰은 또 성매매 알선 자금을 관리한 박씨의 딸(32)과 성매매 여성 김모(33)씨 등 16명을 같은 혐의로 불구속 입건했다.
  예측 : [('경찰', 'OG'), ('박', 'PS'), ('32', 'QT'), ('김', 'PS'), ('33', 'QT'), ('16명', 'QT')]
  정답 : [('경찰', 'OG'), ('박', 'PS'), ('32', 'QT'), ('김', 'PS'), ('33', 'QT'), ('16명', 'QT')]
문장 : B 씨는 사법시험 합격증이 위조된 것인지 알 수 없었고, 아들이 좋은 대학에 들어갈 수 있기만을 바라던터라 A 씨를 더 의심하지 않았다.
  예측 : [('B', 'PS'), ('A', 'PS')]
  정답 : [('B', 'PS'), ('A', 'PS')]
문장 : 수학 A형의 1등급 커트라인은 평균 96점, 수학 B형은 99점이었다.
  예측 : [('A', 'PS'), ('1등', 'QT'), ('96점', 'QT'), ('99점', 'QT')]
  정답 : [('1등급', 'QT'), ('96점', 'QT'), ('99점', 'QT')]
문장 : 정약용은 오늘 독도에서 큰 승리를 거두었다.
  예측 : [('오늘', 'DT'), ('독도', 'LC')]
문장 : 한국철도공사는 오늘부터 독도 지역 조사를 시작한다.
  예측 : [('한국철도공사', 'OG'), ('오늘', 'DT'), ('독도', 'LC')]


> **[결과]** KLUE-NER 학습 21,008문장으로 같은 Bi-LSTM을 5에폭(325초) 학습하고, 검증 5,000문장으로 채점했다.
> - 개체 F1은 micro 0.6509, 유형별 평균(macro) 0.6590으로, 생성 데이터에서 0.77 안팎이던 것보다 낮았다. 토큰 정확도는 0.9350이지만 O가 81.38%라 여기서도 크게 부풀려져 있다.
> - 유형별로는 시간(TI 0.8488)·수량(QT 0.8108)·날짜(DT 0.7764)가 높고, 지명(LC 0.4550)·기관(OG 0.4938)·인명(PS 0.5692)이 낮았다. 숫자에 '시·명·년' 같은 단위가 붙는 개체는 글자 모양만으로 알아보기 쉽지만, 이름은 글자만 봐서는 개체인지 알 수 없기 때문이다.
> - 검증 데이터의 UNK 비율은 0.064%로 3부(5.213%)보다 훨씬 낮았다. 실제 말뭉치에서 어려운 것은 처음 보는 글자가 아니라, 아는 글자로 된 처음 보는 이름이다.
> - 예문에서는 'A형'의 `A`를 인명으로, `1등급`을 `1등`까지만 잡는 오류가 있었다. 3부 문장에서는 `한국철도공사`·`오늘`·`독도`를 모두 찾았지만 `정약용`은 놓쳤다. 규칙을 어긴 I- 태그도 3,096개 나와, 실제 데이터에서도 CRF 제약이 필요하다는 것을 보여 준다.

**심화 과제 서술형 — 무엇이 성능을 바꾸었나**

**[결과]** 같은 3부 데이터에서 개체 F1은 klue/bert-base 0.9953, BIO 제약 CRF 0.8574, 학습만 한 CRF 0.7740, 기본 Bi-LSTM 0.7705, pack 0.7658 순이었다(Bi-LSTM 계열은 시드 3개 평균). 실제 말뭉치 KLUE-NER에서는 같은 Bi-LSTM이 0.6509에 그쳤다.

**[해석]** 처음에는 CRF 층만 얹으면 교안의 설명대로 `O → I` 같은 전이를 스스로 금지하게 될 것이라고 예상했다. 하지만 데이터로만 배운 CRF는 전이 점수가 거의 0에 머물러 기본 모델과 차이가 없었다. 학습 데이터에는 규칙을 어긴 태그열이 하나도 없고, Bi-LSTM이 2~3에폭 만에 학습 데이터를 다 맞히니 CRF가 틀려 보고 벌을 받을 기회가 없었던 것이다. 그래서 작은 데이터에서는 규칙을 직접 넣는 편이 확실했고, 실제로 BIO 제약 하나로 F1이 0.087 올랐다. 다만 제약은 태그열의 앞뒤를 맞출 뿐이다. `B-DAT, I-DAT, B-DAT`처럼 규칙상 허용되는 방식으로 날짜를 쪼갠 `1919년 3월 1일`은 고치지 못했고, 시드에 따라 F1이 0.81~0.90으로 흔들렸다.

`pack_padded_sequence`는 성능을 올리는 장치라기보다 '패딩이 결과에 영향을 주면 안 된다'는 정확성을 지키는 장치였다. 패딩이 늘 문장 뒤에 같은 식으로 붙는 데이터에서는 모델이 그 패턴에 적응해 버려서, 로짓은 3.52까지 달라져도 태그는 바뀌지 않았다. 배치마다 길이 분포가 크게 다르거나, 추론할 때 패딩 길이가 학습 때와 다를 때 의미가 커질 것이다.

BERT가 압도적이었던 이유는 이 실험의 미등록 개체가 BERT에게는 미등록이 아니었기 때문이다. 여기서 OOV는 '이 작은 학습 데이터에 없다'는 뜻이지 '한국어에 없다'는 뜻이 아니고, BERT는 사전학습 때 '정약용'이나 '독도'를 이미 많이 보았다. 사전학습 지식이 OOV 대책으로 그만큼 강력하다는 뜻이지만, 반대로 이 평가는 BERT의 일반화 능력을 실제보다 높게 보여 줄 수 있다. KLUE-NER에서 점수가 떨어진 것은 템플릿 8개로 만든 데이터가 너무 쉬웠다는 증거다. 실제 문장에서는 같은 이름도 다양한 문맥에 나오므로, 문장 틀을 외우는 전략이 통하지 않는다.

**[결론]** 성능을 바꾼 정도는 사전학습 지식, 태그 규칙, 구조 미세조정 순이었다. 실무라면 klue/bert-base 같은 사전학습 모델 위에 BIO 제약 디코딩을 얹는 조합이 맞다고 생각한다. 한계도 있다. Bi-LSTM은 시드 3개, BERT와 KLUE 실험은 시드 1개로만 비교했다. KLUE는 CPU 시간 때문에 5에폭만 학습해 손실이 아직 내려가는 중(0.1504)이었고, BERT를 KLUE로 미세조정하는 실험은 하지 못했다.